
# Pipeline OCR/VLM - Domiciliation des salaires étrangers

**Modèle : Qwen3.6-VL-27B-FP8 sur GPU Domino — version V7.1**

Évolutions V7.1 par rapport à la V7.0 :

1. **Planche « permis de travail » enfin lisible.**
   Cette page contient *deux* documents superposés : le titre de travail
   bilingue en haut (identité, photo, poste, employeur) et la couverture
   « جواز العمل / Permis de Travail » en bas. La V7.0 classait la page sur
   le grand titre du bas et n'extrayait donc que le numéro de série.
   La V7.1 donne la priorité au bloc identité et recadre la page.

2. **Rendu adaptatif.** Les pages dactylographiées sont rendues plus petites
   (moins de tokens, inférence plus rapide) ; la planche permis est
   re-rendue à la demande en haute définition et recadrée sur la zone utile.

3. **Escalade d'extraction.** Si le taux de remplissage d'une page reste
   faible, le pipeline relance automatiquement sur des recadrages plus
   serrés (demi-droite identité, demi-gauche poste/employeur) et fusionne
   les résultats.

4. **Augmentation de salaire.** La ligne
   « Salaire mensuel de base net : X au lieu de Y » du contrat spécifique
   est décomposée en nouveau salaire (X) et ancien salaire (Y), avec calcul
   du montant et du taux d'augmentation.

Architecture inchangée par ailleurs : classification GPU, extraction
spécialisée par type, valeurs brutes + normalisées, un JSON par dossier,
export Excel multi-onglets, mois scindés P1/P2, checkpoint par dossier.

Le pipeline **n'effectue pas les contrôles réglementaires finaux**.
Les règles métier et les décisions restent dans Alteryx.


## 1. Dépendances

In [ ]:

# Installation du runtime minimal compatible Qwen3.6-VL-FP8 (cf. bilans V13c)
# %pip install -q -U 'transformers>=4.57.0' accelerate

import sys
from importlib import metadata

REQUIRED_PACKAGES = {
    "torch": "2.0",
    "transformers": "4.57",
    "accelerate": "0.30",
    "PyMuPDF": "1.23",
    "Pillow": "9.0",
    "openpyxl": "3.1",
    "pandas": "1.5",
    "psutil": "5.9",
}

print("Python :", sys.version.replace("\n", " "))
print("\nPackages détectés :")
missing = []
for package_name, minimum in REQUIRED_PACKAGES.items():
    try:
        version = metadata.version(package_name)
        print(f"  {package_name:15s} {version:12s} | minimum conseillé {minimum}")
    except metadata.PackageNotFoundError:
        missing.append(package_name)
        print(f"  {package_name:15s} ABSENT")

if missing:
    raise RuntimeError(
        "Packages manquants : " + ", ".join(missing) +
        ". Installer uniquement ces packages dans l'environnement Domino."
    )

print("\n✅ Vérification des packages terminée sans modification de l'environnement")


## 2. Imports

In [ ]:

import gc
import hashlib
import json
import math
import re
import sys
import time
import calendar
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path

import fitz
import numpy as np
import pandas as pd
import psutil
import torch
from PIL import Image
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from transformers import AutoProcessor, AutoModelForImageTextToText

print("✅ Imports OK")
print("Python       :", sys.version.split()[0])
print("PyMuPDF     :", fitz.__doc__.splitlines()[0] if fitz.__doc__ else "chargé")
print("Torch       :", torch.__version__)
print("CUDA dispo  :", torch.cuda.is_available())
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")


## 3. Configuration

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---------------------------------------------------------------------
# Rendu adaptatif
# ---------------------------------------------------------------------
# Les contrats et engagements sont dactylographiés : un rendu modéré suffit
# et divise le nombre de tokens image par rapport à la V7.0.
PDF_ZOOM = 3.0
IMAGE_MAX_SIZE = 1400

# La planche « permis de travail » est un scan dégradé, bilingue, avec des
# valeurs manuscrites ou tamponnées : elle est re-rendue à la demande.
PDF_ZOOM_HAUTE_DEF = 4.5
IMAGE_MAX_SIZE_HAUTE_DEF = 2200

MIN_PIXELS = 4 * 32 * 32
# Le plafond du processor doit couvrir le rendu haute définition.
MAX_PIXELS = 2600 * 32 * 32

BLANK_THRESHOLD = 0.995
GPU_BATCH_SIZE_CLASSIFICATION = 2
GPU_BATCH_SIZE_EXTRACTION = 1
MAX_NEW_TOKENS_CLASSIFICATION = 100
MAX_NEW_TOKENS_EXTRACTION = 1700

# ---------------------------------------------------------------------
# Escalade d'extraction sur les pages difficiles
# ---------------------------------------------------------------------
# Types de page traités d'emblée en haute définition recadrée.
TYPES_HAUTE_DEFINITION = {"TITRE_TRAVAIL", "PERMIS_TRAVAIL_COUVERTURE"}

# Taux de remplissage au-dessus duquel on arrête d'escalader.
SEUIL_REMPLISSAGE_OK = 0.70

INPUT_DIR = Path('/mnt/data/domiciliations_in')
OUTPUT_DIR = Path('/mnt/data/domiciliations_out')
JSON_DIR = OUTPUT_DIR / 'json_dossiers'
LOG_PATH = OUTPUT_DIR / 'pipeline_domiciliations.log'
EXCEL_PATH = OUTPUT_DIR / f"domiciliations_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
MASTER_JSON_PATH = OUTPUT_DIR / 'domiciliations_master.json'
PRORATA_MODE = 'CALENDAR_DAYS'
GENERER_MOIS_COMPLETS = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f'Device          : {DEVICE}')
print(f'PDFs détectés   : {len(pdfs)}')
print(f'Entrée          : {INPUT_DIR}')
print(f'Sortie          : {OUTPUT_DIR}')
print(f'Prorata retenu  : {PRORATA_MODE}')
CLASSIFICATION_THRESHOLD = 0.80

PIPELINE_VERSION = "DOM_V7_1_QWEN3_6_VL_27B_FP8"
DOM_REFERENCE_EXCEL = Path("/mnt/data/fichier_domiciliations.xlsx")
PREDOM_REFERENCE_EXCEL = Path("/mnt/data/fichier_predomiciliations.xlsx")
NAME_MATCH_THRESHOLD = 0.86
DATE_TOLERANCE_DAYS = 5
EXCEL_AMOUNT_FORMAT = "0.00"
AMOUNT_FIELDS = {
    "DOM_SALAIRE_NET_MENSUEL", "DOM_PART_TRANSFERABLE",
    "DOM_MONTANT_TOTAL_DOMICILIE", "CTR_SALAIRE_BRUT",
    "CTR_SALAIRE_NET", "CTS_SALAIRE_NET",
    "CTS_SALAIRE_NET_ANCIEN",
    "CTS_PART_TRANSFERABLE", "CTS_PART_PAYABLE_DZD",
}


## 4. Chargement du modèle Qwen3.6-VL-27B-FP8

In [ ]:

if DEVICE != "cuda":
    raise RuntimeError("Ce pipeline nécessite un GPU CUDA.")

torch.backends.cuda.matmul.allow_tf32 = True

print("Chargement du processor...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = "left"

print("Chargement du modèle FP8...")
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()

print(f"✅ Modèle chargé en {time.time() - t0:.1f}s | dtype=bfloat16 (FP8 déquantifié)")
print(f"VRAM allouée : {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 5. Utilitaires PDF, image et JSON

In [ ]:

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)


def white_ratio(image):
    arr = np.array(image.convert("L"))
    return float((arr > 245).sum() / arr.size)


def is_blank(image, threshold=BLANK_THRESHOLD):
    return white_ratio(image) >= threshold


def pdf_to_pages(path, zoom=PDF_ZOOM):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"PDF introuvable : {path}")
    if path.stat().st_size == 0:
        raise ValueError(f"PDF vide : {path}")

    pages = []
    doc = fitz.open(str(path))
    try:
        page_count = int(doc.page_count)
        if page_count <= 0:
            raise ValueError(f"PyMuPDF ne détecte aucune page dans : {path.name}")

        matrix = fitz.Matrix(zoom, zoom)
        for i in range(page_count):
            page = doc.load_page(i)
            pix = page.get_pixmap(matrix=matrix, alpha=False)

            if pix.width <= 0 or pix.height <= 0 or not pix.samples:
                raise ValueError(
                    f"Rendu image vide : {path.name}, page {i + 1}"
                )

            img = Image.frombytes(
                "RGB",
                (pix.width, pix.height),
                pix.samples
            )
            img = resize_image(img)

            pages.append({
                "index": i,
                "page_num": i + 1,
                "image": img,
                "width": img.width,
                "height": img.height,
                "white_ratio": round(white_ratio(img), 6),
            })
    finally:
        doc.close()

    if len(pages) != page_count:
        raise RuntimeError(
            f"Conversion incomplète de {path.name}: "
            f"{len(pages)} image(s) pour {page_count} page(s)"
        )
    return pages


def parse_json_response(text):
    if not text:
        return {}

    clean = str(text).strip()
    clean = re.sub(r"^```(?:json)?", "", clean, flags=re.I).strip()
    clean = re.sub(r"```$", "", clean).strip()

    match = re.search(r"\{.*\}", clean, flags=re.S)
    if not match:
        return {}

    candidate = match.group(0)
    attempts = [
        candidate,
        re.sub(r",\s*([}\]])", r"\1", candidate),
    ]

    for attempt in attempts:
        try:
            parsed = json.loads(attempt)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            continue
    return {}


def log(message):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


print("✅ Utilitaires PDF/JSON OK")


# ---------------------------------------------------------------------
# Rendu haute définition et recadrage (planche « permis de travail »)
# ---------------------------------------------------------------------

def crop_region(image, haut=0.0, bas=1.0, gauche=0.0, droite=1.0):
    """
    Recadre une image par fractions de sa hauteur et de sa largeur.

    Les fractions sont exprimées entre 0.0 et 1.0 :
    crop_region(img, 0.0, 0.58) garde les 58 % supérieurs de la page.
    """
    largeur, hauteur = image.size

    x0 = int(max(0.0, min(1.0, gauche)) * largeur)
    x1 = int(max(0.0, min(1.0, droite)) * largeur)
    y0 = int(max(0.0, min(1.0, haut)) * hauteur)
    y1 = int(max(0.0, min(1.0, bas)) * hauteur)

    if x1 <= x0 or y1 <= y0:
        return image

    return image.crop((x0, y0, x1, y1))


def render_page_region(
    pdf_path,
    page_index,
    zoom=PDF_ZOOM_HAUTE_DEF,
    max_side=IMAGE_MAX_SIZE_HAUTE_DEF,
    crop=None,
):
    """
    Re-rend une page du PDF en haute définition, éventuellement recadrée.

    Le recadrage est appliqué AVANT le redimensionnement : la zone utile
    récupère donc toute la résolution disponible, au lieu d'être diluée
    dans une page entière réduite à IMAGE_MAX_SIZE.
    """
    pdf_path = Path(pdf_path)
    doc = fitz.open(str(pdf_path))
    try:
        page = doc.load_page(int(page_index))
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)

        if pix.width <= 0 or pix.height <= 0 or not pix.samples:
            raise ValueError(
                f"Rendu haute définition vide : {pdf_path.name}, "
                f"page {int(page_index) + 1}"
            )

        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    finally:
        doc.close()

    if crop:
        img = crop_region(img, *crop)

    return resize_image(img, max_side=max_side)


def taux_remplissage(data, champs_attendus):
    """
    Part des champs attendus effectivement renseignés.
    Sert de critère d'escalade vers un recadrage plus serré.
    """
    if not champs_attendus:
        return 1.0

    remplis = sum(
        1
        for champ in champs_attendus
        if (data or {}).get(champ) not in (None, "")
    )
    return round(remplis / len(champs_attendus), 4)


print("✅ Rendu haute définition et recadrage OK")


# ---------------------------------------------------------------------
# V7.1 : détection automatique de la frontière sur la planche permis
# ---------------------------------------------------------------------
# La planche « permis de travail » superpose deux documents. La position
# de la coupure varie d'un scan à l'autre (marges, cadrage du photocopieur).
# On la localise en cherchant la plus large bande horizontale sans encre
# dans la zone médiane, au lieu de figer une fraction dans le code.

FRONTIERE_ZONE_RECHERCHE = (0.28, 0.66)   # plage explorée
FRONTIERE_SEUIL_ENCRE = 0.004             # densité en dessous de laquelle
                                          # une ligne est considérée vide
FRONTIERE_DEFAUT = 0.47                   # repli si aucune bande nette


def detecter_frontiere_documents(
    image,
    zone=FRONTIERE_ZONE_RECHERCHE,
    seuil_encre=FRONTIERE_SEUIL_ENCRE,
    defaut=FRONTIERE_DEFAUT,
):
    """
    Renvoie la fraction de hauteur séparant les deux documents d'une planche.

    Retourne `defaut` si aucune bande blanche franche n'est trouvée, ce qui
    laisse le pipeline fonctionner sur une planche atypique.
    """
    try:
        arr = np.array(image.convert("L"))
    except Exception:
        return defaut

    hauteur = arr.shape[0]
    if hauteur < 10:
        return defaut

    densite_encre = (arr < 200).sum(axis=1) / max(arr.shape[1], 1)

    y_min = int(max(0.0, zone[0]) * hauteur)
    y_max = int(min(1.0, zone[1]) * hauteur)
    if y_max <= y_min:
        return defaut

    bandes = []
    debut = None
    for y in range(y_min, y_max):
        vide = densite_encre[y] < seuil_encre
        if vide and debut is None:
            debut = y
        elif not vide and debut is not None:
            bandes.append((debut, y))
            debut = None
    if debut is not None:
        bandes.append((debut, y_max))

    if not bandes:
        return defaut

    debut, fin = max(bandes, key=lambda b: b[1] - b[0])

    # Une bande trop fine n'est qu'un interligne, pas une coupure.
    if (fin - debut) / hauteur < 0.015:
        return defaut

    return round((debut + fin) / 2 / hauteur, 4)


def crops_planche_permis(image, marge=0.02):
    """
    Construit les recadrages de la planche à partir de la frontière détectée.

    Retourne un dictionnaire de tuples (haut, bas, gauche, droite) :
      - titre           : bloc titre de travail (identité + poste) ;
      - colonne_identite: moitié droite du bloc titre (Nom, Prénom, dates) ;
      - colonne_poste   : moitié gauche du bloc titre (poste, employeur) ;
      - couverture      : bloc bas (Permis de Travail, extraits de loi).
    """
    frontiere = detecter_frontiere_documents(image)

    bas_titre = min(1.0, frontiere + marge)
    haut_couverture = max(0.0, frontiere - marge)

    return {
        "frontiere": frontiere,
        "titre": (0.00, bas_titre, 0.00, 1.00),
        "colonne_identite": (0.00, bas_titre, 0.44, 1.00),
        "colonne_poste": (0.00, bas_titre, 0.00, 0.56),
        "couverture": (haut_couverture, 1.00, 0.00, 1.00),
    }


print("✅ Détection automatique de la frontière de planche OK")


In [ ]:

def canonical_checkpoint_path(pdf_path):
    """
    Un seul fichier JSON par PDF :
    <nom_du_pdf_sans_extension>.json
    """
    return JSON_DIR / f"{pdf_path.stem}.json"


def checkpoint_is_complete(dossier, pdf_path):
    """
    Un checkpoint est réutilisable seulement s'il correspond au PDF
    et contient une extraction complète avec au moins une page.
    """
    if not isinstance(dossier, dict):
        return False

    stats = dossier.get("stats") or {}
    page_records = dossier.get("page_records") or []

    if dossier.get("source_file") != pdf_path.name:
        return False
    if int(stats.get("pages", 0) or 0) <= 0:
        return False
    if not page_records:
        return False

    # Vérification forte par empreinte SHA-256.
    stored_hash = dossier.get("source_sha256")
    if not stored_hash:
        return False

    try:
        return stored_hash == sha256_file(pdf_path)
    except Exception:
        return False


def load_existing_checkpoint(pdf_path):
    """
    Cherche d'abord le JSON canonique, puis les anciens JSON suffixés
    par l'empreinte. Si un ancien checkpoint valide est trouvé, il est
    migré vers le nom canonique afin de conserver un seul JSON par PDF.
    """
    canonical = canonical_checkpoint_path(pdf_path)
    candidates = [canonical] + sorted(
        JSON_DIR.glob(f"{pdf_path.stem}__*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    seen = set()
    for candidate in candidates:
        if candidate in seen or not candidate.exists():
            continue
        seen.add(candidate)

        try:
            dossier = json.loads(candidate.read_text(encoding="utf-8"))
        except Exception:
            continue

        if not checkpoint_is_complete(dossier, pdf_path):
            continue

        # Migration vers un seul JSON canonique.
        if candidate != canonical:
            canonical.write_text(
                json.dumps(dossier, ensure_ascii=False, indent=2, default=str),
                encoding="utf-8",
            )

        # Suppression des anciens doublons JSON du même PDF.
        for duplicate in JSON_DIR.glob(f"{pdf_path.stem}__*.json"):
            if duplicate.exists():
                duplicate.unlink()

        return dossier

    return None


def enrich_dossier_row_with_stats(dossier, statut_traitement):
    """
    Ajoute au niveau dossier les indicateurs visibles dans Excel.
    """
    row = dossier.get("dossier_row") or {}
    stats = dossier.get("stats") or {}

    row.update({
        "STATUT_TRAITEMENT_PIPELINE": statut_traitement,
        "TEMPS_ECOULE_DOSSIER_S": round(float(stats.get("elapsed_s", 0) or 0), 2),
        "TOKENS_IN_DOSSIER": int(stats.get("tokens_in", 0) or 0),
        "TOKENS_OUT_DOSSIER": int(stats.get("tokens_out", 0) or 0),
        "TOKENS_TOTAL_DOSSIER": int(stats.get("tokens_total", 0) or 0),
    })

    dossier["dossier_row"] = row
    return dossier


In [ ]:

def format_duration(seconds):
    seconds = max(0, int(round(float(seconds or 0))))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def print_pipeline_header(total_pdfs):
    print(
        f"\n{PIPELINE_VERSION} | {total_pdfs} PDF détecté(s)\n",
        flush=True,
    )


def print_compact_progress(
    position,
    total,
    pdf_name,
    pages,
    skipped,
    tokens_in,
    tokens_out,
    elapsed_s,
    pipeline_start,
):
    elapsed_global = time.time() - pipeline_start
    avg = elapsed_global / max(position, 1)
    eta = avg * max(total - position, 0)
    status = "SKIP" if skipped else "TRAITÉ"

    print(
        f"[{position}/{total}] "
        f"{pdf_name} | "
        f"pages={int(pages or 0)} | "
        f"{status} | "
        f"IN={int(tokens_in or 0):,} | "
        f"OUT={int(tokens_out or 0):,} | "
        f"{float(elapsed_s or 0):.2f}s | "
        f"ETA={format_duration(eta)}",
        flush=True,
    )


## 6. Inférence GPU batch

In [ ]:

def apply_template(messages):
    """Qwen3 : désactive le mode 'thinking' si supporté."""
    try:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def ask_single(prompt, image, max_new_tokens):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    text_in = apply_template(messages)
    inputs = processor(
        text=[text_in],
        images=[image],
        return_tensors="pt",
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()

    generated = out[0][inputs["input_ids"].shape[1]:]
    text = processor.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    return {
        "text": text,
        "tokens_in": int(inputs["input_ids"].shape[1]),
        "tokens_out": int(len(generated)),
        "elapsed_s": round(time.time() - t0, 3),
    }


def ask_batch(prompt, images, max_new_tokens):
    if not images:
        return []
    if len(images) == 1:
        return [ask_single(prompt, images[0], max_new_tokens)]

    texts_in = []
    for image in images:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]
        texts_in.append(apply_template(messages))

    inputs = processor(
        text=texts_in,
        images=images,
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()

    if out.shape[0] != len(images):
        raise RuntimeError(
            f"Réponses VLM incohérentes : {out.shape[0]} sortie(s) "
            f"pour {len(images)} image(s)"
        )

    elapsed = time.time() - t0
    input_width = inputs["input_ids"].shape[1]
    attention_mask = inputs.get("attention_mask")
    results = []

    for i in range(len(images)):
        generated = out[i][input_width:]
        text = processor.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        tokens_in = (
            int(attention_mask[i].sum().item())
            if attention_mask is not None
            else int(input_width)
        )
        results.append({
            "text": text,
            "tokens_in": tokens_in,
            "tokens_out": int(len(generated)),
            "elapsed_s": round(elapsed / len(images), 3),
        })

    return results


print("✅ Inférence single/batch OK")


## 7. Prompts de classification et d’extraction

In [ ]:

PROMPT_CLASSIFICATION = """
Analyse le titre, les en-têtes, la mise en page et les blocs visuels
de cette page.

Classe la page dans exactement une seule catégorie :

- ENGAGEMENT_DOMICILIATION
- CONTRAT_TRAVAIL
- CONTRAT_SPECIFIQUE
- TITRE_TRAVAIL
- PERMIS_TRAVAIL_COUVERTURE
- AUTRE

RÈGLE DE PRIORITÉ ABSOLUE :
Certaines pages contiennent DEUX documents superposés : un titre de
travail bilingue dans la moitié haute et une couverture de permis de
travail dans la moitié basse.
Dans ce cas, classer TOUJOURS la page en TITRE_TRAVAIL.
Le grand titre « جواز العمل / Permis de Travail » de la moitié basse ne
doit JAMAIS l'emporter sur un bloc d'identité présent dans la moitié
haute.

Règles de classification :

- ENGAGEMENT_DOMICILIATION :
  le titre contient « ENGAGEMENT DE DOMICILIATION »
  ou « CONTRAT DES SALARIES ETRANGERS ».

- CONTRAT_TRAVAIL :
  le titre contient « CONTRAT DE TRAVAIL A DUREE DETERMINEE ».

- CONTRAT_SPECIFIQUE :
  le titre contient « CONTRAT DE TRAVAIL SPECIFIQUE
  A LA MAIN D'OEUVRE ETRANGERE ».

- TITRE_TRAVAIL :
  la page contient un bloc d'identité du travailleur, reconnaissable à
  AU MOINS DEUX des éléments suivants :
    - une photographie d'identité ;
    - les libellés « Nom » et « Prénom » suivis de valeurs ;
    - les libellés « Date de naissance » / « Lieu de naissance » ;
    - le libellé « Date d'entrée en Algérie » ;
    - des libellés arabes d'identité (اللقب، الإسم، تاريخ الإزدياد).
  Cette catégorie s'applique même si la page comporte aussi des cachets,
  un QR code, du texte de loi ou un second document en dessous.

- PERMIS_TRAVAIL_COUVERTURE :
  UNIQUEMENT si la page ne contient AUCUN bloc d'identité du travailleur
  et se limite au titre « Permis de Travail », au numéro de série et aux
  extraits de loi.

- AUTRE :
  aucun type ne correspond clairement.

Ne te base jamais uniquement sur le numéro de page.

Retourne uniquement ce JSON :
{
  "type_document": "TYPE",
  "confidence": 0.00,
  "titre_detecte": "TITRE BRUT OU null",
  "bloc_identite_present": true
}
"""

COMMON_RAW_RULES = """
Tu analyses une seule image, qui peut être une page entière ou un
recadrage d'une page.

RÈGLES OBLIGATOIRES :
1. Extraire uniquement les champs demandés.
2. Pour chaque champ, rechercher le libellé indiqué.
3. Recopier uniquement la valeur située juste après le libellé :
   - sur la même ligne ;
   - ou immédiatement sur la ligne suivante si la valeur continue.
4. Conserver la valeur exactement comme elle apparaît :
   espaces, ponctuation, séparateurs, format de date et format de montant.
5. Ne corrige pas l'orthographe.
6. Ne normalise pas les dates.
7. Ne normalise pas les montants.
8. Ne sépare pas automatiquement le nom et le prénom.
9. Ne complète pas une valeur partiellement lisible.
10. N'utilise aucune valeur provenant d'une autre page.
11. Si le libellé est absent ou la valeur illisible, retourne null.
12. N'invente jamais une valeur.
13. Retourne uniquement un objet JSON valide, sans commentaire.
14. Un champ absent du recadrage que tu analyses doit valoir null.
    Ne devine pas ce qui se trouve hors de l'image.
"""

PROMPT_ENGAGEMENT = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION

Extrais exactement les clés suivantes :

{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_ADRESSE_CLIENT": null,
  "DOM_AGENCE_DOMICILIATAIRE": null,
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null,
  "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": null,
  "DOM_ADRESSE_EMPLOYEUR": null,
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null,
  "DOM_DATE_SIGNATURE": null
}

Libellés et règles :

- DOM_NOM_RAISON_SOCIAL_CLIENT :
  valeur après « Nom et raison sociale ».
  La valeur peut être répartie en deux colonnes (nom puis prénom) :
  recopier les deux, séparés par un espace.

- DOM_COMPTE_LOCAL :
  valeur après « N de compte » ou « N° de compte ».

- DOM_ADRESSE_CLIENT :
  valeur après la première occurrence de « Adresse »
  dans la section « Identification du client ».

- DOM_AGENCE_DOMICILIATAIRE :
  valeur après « Agence domiciliataire », en haut à droite.

- DOM_NUMERO_CONTRAT :
  valeur après « Numéro du contrat ».

- DOM_DUREE_CONTRAT_MOIS :
  valeur après « Durée du contrat » ou « Duré du contrat ».

- DOM_DATE_DEBUT_CONTRAT :
  valeur après « Date de début de contrat ».

- DOM_DATE_FIN_CONTRAT :
  valeur après « Date de fin de contrat ».

- DOM_NOM_RAISON_SOCIAL_EMPLOYEUR :
  valeur après « Nom et raison sociale de L'Employeur ».

- DOM_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de L'Employeur ».
  Continuer sur la ligne suivante si l'adresse se poursuit.

- DOM_SALAIRE_NET_MENSUEL :
  valeur après « Salaire net mensuel ».

- DOM_PART_TRANSFERABLE :
  valeur après « Montant de la part transférable ».

- DOM_TAUX_TRANSFERABLE :
  valeur après « Pourcentage en regard du salaire net mensuel ».

- DOM_MONTANT_TOTAL_DOMICILIE :
  valeur après « Montant domicilié en DZD ».
  Ce champ est souvent laissé vide : retourner null dans ce cas.

- DOM_DATE_SIGNATURE :
  date manuscrite ou imprimée située près de la mention
  « lu et approuvé », en bas de page.
"""

PROMPT_CONTRAT = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_TRAVAIL

Extrais exactement les clés suivantes :

{
  "CTR_REFERENCE_DOCUMENT": null,
  "CTR_TYPE": null,
  "CTR_EMPLOYEUR": null,
  "CTR_ACTIVITE_EMPLOYEUR": null,
  "CTR_DUREE_MOIS": null,
  "CTR_DATE_DEBUT_CONTRAT": null,
  "CTR_POSTE": null,
  "CTR_NOM_PRENOM_TRAVAILLEUR": null,
  "CTR_PERE_NOM_PRENOM": null,
  "CTR_MERE_NOM_PRENOM": null,
  "CTR_NATIONALITE": null,
  "CTR_DATE_NAISSANCE": null,
  "CTR_LIEU_PAYS_NAISSANCE": null,
  "CTR_ADRESSE_ALGERIE": null,
  "CTR_QUALIFICATION": null,
  "CTR_NUMERO_PERMIS_TRAVAIL": null,
  "CTR_DATE_DELIVRANCE_PERMIS": null,
  "CTR_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTR_DATE_FIN_VALIDITE_PERMIS": null,
  "CTR_SALAIRE_BRUT": null,
  "CTR_SALAIRE_NET": null,
  "CTR_AFFILIATION_SS": null,
  "CTR_NUMERO_EMPLOYEUR": null,
  "CTR_DATE_SIGNATURE": null,
  "CTR_REFERENCE_DOMICILIATION": null,
  "CTR_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTR_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTR_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :

- CTR_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche,
  par exemple « TR-6166 ».

- CTR_TYPE :
  titre complet du document.

- CTR_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTR_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTR_DUREE_MOIS :
  valeur après « pour une durée de : »
  et avant « à compter du ».

- CTR_DATE_DEBUT_CONTRAT :
  valeur après « à compter du : ».

- CTR_POSTE :
  valeur après « En qualité de : ».

- CTR_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTR_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTR_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTR_NATIONALITE :
  valeur après « Nationalité : ».

- CTR_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTR_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTR_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTR_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTR_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTR_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTR_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « Valable du ».

- CTR_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

- CTR_SALAIRE_BRUT :
  valeur après « Montant du salaire mensuel brut : ».

- CTR_SALAIRE_NET :
  valeur après « Montant du salaire mensuel net : ».

- CTR_AFFILIATION_SS :
  valeur après « Affiliation à la sécurité sociale : ».

- CTR_NUMERO_EMPLOYEUR :
  valeur après « Employeur : ».

- CTR_DATE_SIGNATURE :
  date après « Fait à : Bethioua, le ».

- CTR_REFERENCE_DOMICILIATION :
  dans le cachet « DOMICILIATION IMPORT »,
  recopier les cinq cases dans l'ordre
  et les séparer par « | ».
  Exemple : 271901|2026.1|40|00119|DZD
  null si ce cachet est absent de la page.

Contrôles visuels :
- CTR_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature du Travailleur Etranger », sinon false.

- CTR_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature de l'Employeur », sinon false.

- CTR_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone
  « Signature de l'Employeur », sinon false.
"""

PROMPT_CONTRAT_SPECIFIQUE = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_SPECIFIQUE

Extrais exactement les clés suivantes :

{
  "CTS_REFERENCE_DOCUMENT": null,
  "CTS_SAP_ID": null,
  "CTS_EMPLOYEUR": null,
  "CTS_ACTIVITE_EMPLOYEUR": null,
  "CTS_DUREE_MOIS": null,
  "CTS_DATE_DEBUT_CONTRAT": null,
  "CTS_POSTE": null,
  "CTS_NOM_PRENOM_TRAVAILLEUR": null,
  "CTS_PERE_NOM_PRENOM": null,
  "CTS_MERE_NOM_PRENOM": null,
  "CTS_NATIONALITE": null,
  "CTS_DATE_NAISSANCE": null,
  "CTS_LIEU_PAYS_NAISSANCE": null,
  "CTS_ADRESSE_ALGERIE": null,
  "CTS_QUALIFICATION": null,
  "CTS_NUMERO_PERMIS_TRAVAIL": null,
  "CTS_DATE_DELIVRANCE_PERMIS": null,
  "CTS_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTS_DATE_FIN_VALIDITE_PERMIS": null,
  "CTS_LIGNE_SALAIRE_BRUTE": null,
  "CTS_SALAIRE_NET": null,
  "CTS_SALAIRE_NET_ANCIEN": null,
  "CTS_MENTION_AU_LIEU_DE_PRESENTE": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_NUMERO_SS_PAYS_ORIGINE": null,
  "CTS_NUMERO_SS_ALGERIE": null,
  "CTS_DATE_DOCUMENT": null,
  "CTS_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTS_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTS_CACHET_EMPLOYEUR_PRESENT": null,
  "CTS_VISA_INSPECTION_TRAVAIL_PRESENT": null
}

Libellés et règles :

- CTS_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche, par exemple « TR-6166 ».

- CTS_SAP_ID :
  valeur après « SAP id - » en haut de page.

- CTS_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTS_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTS_DUREE_MOIS :
  valeur après « pour une durée de : ».

- CTS_DATE_DEBUT_CONTRAT :
  valeur après « A compter du : ».

- CTS_POSTE :
  valeur après « en qualité de : ».

- CTS_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTS_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTS_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTS_NATIONALITE :
  valeur après « Nationalité : ».

- CTS_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTS_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTS_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTS_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTS_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTS_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTS_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « valable du ».

- CTS_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

--- LIGNE DE SALAIRE : RÈGLE PARTICULIÈRE ---

Cette ligne peut prendre DEUX formes :

  Forme A (salaire inchangé) :
    « Salaire mensuel de base net : 506,471.38 »

  Forme B (augmentation de salaire) :
    « Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29 »

- CTS_LIGNE_SALAIRE_BRUTE :
  recopier la ligne ENTIÈRE telle qu'elle apparaît, du libellé
  « Salaire mensuel de base net » jusqu'à la fin de la ligne,
  sans rien retirer.

- CTS_SALAIRE_NET :
  le PREMIER montant de cette ligne, c'est-à-dire celui situé
  immédiatement après « Salaire mensuel de base net : ».
  En forme B, c'est le NOUVEAU salaire, celui placé AVANT
  « au lieu de ». Ne jamais recopier « au lieu de » ni ce qui suit.

- CTS_SALAIRE_NET_ANCIEN :
  le SECOND montant de cette ligne, celui placé APRÈS
  « au lieu de ». C'est l'ANCIEN salaire.
  null si la mention « au lieu de » est absente de cette ligne.

- CTS_MENTION_AU_LIEU_DE_PRESENTE :
  true si la ligne de salaire contient « au lieu de », sinon false.

--- SUITE ---

- CTS_PART_TRANSFERABLE :
  valeur après « La part transférable : ».

- CTS_PART_PAYABLE_DZD :
  valeur après « La part payable en dinars algérien : ».

- CTS_NUMERO_SS_PAYS_ORIGINE :
  valeur après « Dans le pays d'origine : ».

- CTS_NUMERO_SS_ALGERIE :
  valeur après « En Algérie : ».

- CTS_DATE_DOCUMENT :
  date après « Fait à : Bethioua, le ».

Contrôles visuels :
- CTS_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature du Travailleur Etranger ».

- CTS_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature de l'Employeur ».

- CTS_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone employeur.

- CTS_VISA_INSPECTION_TRAVAIL_PRESENT :
  true si le bas de page comporte un cachet ou une mention manuscrite
  près de « Le présent contrat a été visé par nous ».
"""

PROMPT_TITRE_TRAVAIL = COMMON_RAW_RULES + """
TYPE ATTENDU : TITRE_TRAVAIL

DESCRIPTION DE LA PAGE :
Il s'agit d'un titre de travail algérien, bilingue arabe/français,
photocopié en noir et blanc. La qualité est dégradée, les valeurs sont
souvent inscrites sur des lignes de pointillés, et des cachets ronds
peuvent recouvrir partiellement le texte.

MISE EN PAGE — À LIRE ATTENTIVEMENT :
Le document est organisé en DEUX COLONNES.

  COLONNE DE GAUCHE — le poste et l'employeur.
  Chaque ligne suit le schéma :
      libellé français ..... valeur .....        libellé arabe
  Le libellé arabe est collé au bord DROIT de la colonne gauche.
  La valeur se trouve ENTRE le libellé français et le libellé arabe.
  Libellés : « Durée », « Du », « Au », « Lieu de travail »,
  « Nom de l'organisme employeur », « Adresse de l'organisme employeur »,
  « Fait à », « Le ».

  COLONNE DE DROITE — l'identité du travailleur, à côté de la photo.
  Chaque ligne suit le schéma :
      libellé français ..... valeur .....        libellé arabe
  Libellés : « Nom », « Prénom », « Date de naissance »,
  « Lieu de naissance », « Pays », « Nationalité », « Qualification »,
  « Date d'entrée en Algérie ».

RÈGLE CRITIQUE :
Ne JAMAIS confondre un libellé arabe avec une valeur.
La valeur est toujours le texte latin situé sur les pointillés,
entre le libellé français et le libellé arabe.
Si une ligne ne contient que des libellés et des pointillés vides,
retourner null pour ce champ.

Extrais exactement :

{
  "TTR_NUMERO_PERMIS": null,
  "TTR_NUMERO_MANUSCRIT": null,
  "TTR_POSTE": null,
  "TTR_DUREE": null,
  "TTR_DATE_DEBUT": null,
  "TTR_DATE_FIN": null,
  "TTR_LIEU_TRAVAIL": null,
  "TTR_EMPLOYEUR": null,
  "TTR_ADRESSE_EMPLOYEUR": null,
  "TTR_FAIT_A": null,
  "TTR_DATE_DELIVRANCE": null,
  "TTR_NOM": null,
  "TTR_PRENOM": null,
  "TTR_DATE_NAISSANCE": null,
  "TTR_LIEU_NAISSANCE": null,
  "TTR_PAYS": null,
  "TTR_NATIONALITE": null,
  "TTR_QUALIFICATION": null,
  "TTR_DATE_ENTREE_ALGERIE": null,
  "TTR_PHOTO_PRESENTE": null,
  "TTR_CACHET_PRESENT": null
}

Libellés et règles :

- TTR_NUMERO_PERMIS :
  référence encadrée en haut à gauche, de la forme
  « ( R ) 21-00002974 / 31-25-001448 ».
  Recopier la référence complète, y compris les deux parties
  séparées par « / ». Ignorer les parenthèses et la lettre isolée.

- TTR_NUMERO_MANUSCRIT :
  nombre manuscrit inscrit juste sous la référence encadrée,
  par exemple « 6466 ». null si absent.

- TTR_POSTE :
  texte situé sous la phrase
  « Le titulaire du présent permis de travail est autorisé à occuper
  le poste de travail de ».
  Ce texte peut s'étendre sur deux ou trois lignes de pointillés :
  recopier l'ensemble, en séparant les fragments par un espace.

- TTR_DUREE :
  valeur après « Durée », par exemple « 2 ANS, 0 JOURS ».

- TTR_DATE_DEBUT :
  valeur après « Du », sur la ligne portant le libellé arabe
  « إبتداء من ».

- TTR_DATE_FIN :
  valeur après « Au », sur la ligne portant le libellé arabe « إلى ».
  Attention : ce libellé est « Au », pas « Fin de travail ».

- TTR_LIEU_TRAVAIL :
  valeur après « Lieu de travail ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_EMPLOYEUR :
  valeur après « Nom de l'organisme employeur ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de l'organisme employeur ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_FAIT_A :
  valeur après « Fait à ».

- TTR_DATE_DELIVRANCE :
  valeur après « Le », sous « Fait à ».
  Cette date est fréquemment recouverte par un cachet rond :
  si elle reste illisible, retourner null plutôt que de deviner.

- TTR_NOM :
  valeur après « Nom », ligne portant le libellé arabe « اللقب ».
  C'est le nom de famille seul.

- TTR_PRENOM :
  valeur après « Prénom », ligne portant le libellé arabe « الإسم ».

- TTR_DATE_NAISSANCE :
  valeur après « Date de naissance »,
  ligne portant le libellé arabe « تاريخ الإزدياد ».

- TTR_LIEU_NAISSANCE :
  valeur après « Lieu de naissance »,
  ligne portant le libellé arabe « مكان الإزدياد ».
  La valeur peut associer une ville et un pays séparés par « / » :
  recopier l'ensemble.

- TTR_PAYS :
  valeur après « Pays », ligne portant le libellé arabe « البلد ».

- TTR_NATIONALITE :
  valeur après « Nationalité »,
  ligne portant le libellé arabe « الجنسية ».

- TTR_QUALIFICATION :
  valeur après « Qualification »,
  ligne portant le libellé arabe « التأهيل ».
  La valeur peut être coupée en fin de ligne et se poursuivre sur la
  ligne suivante : recopier l'ensemble sans ajouter d'espace au point
  de coupure si le mot est manifestement scindé.

- TTR_DATE_ENTREE_ALGERIE :
  valeur après « Date d'entrée en Algérie »,
  ligne portant le libellé arabe « تاريخ الدخول إلى الجزائر ».

Contrôles visuels :
- TTR_PHOTO_PRESENTE :
  true si une photographie d'identité est visible en haut à droite.

- TTR_CACHET_PRESENT :
  true si au moins un cachet rond est visible sur le document.
"""

PROMPT_PERMIS_COUVERTURE = COMMON_RAW_RULES + """
TYPE ATTENDU : PERMIS_TRAVAIL_COUVERTURE

Cette image correspond à la couverture du permis de travail :
titre « جواز العمل / Permis de Travail », extraits de loi et cachet
de la Direction de l'Emploi de la Wilaya.

Extrais exactement :

{
  "PTR_NUMERO_SERIE": null,
  "PTR_WILAYA": null,
  "PTR_CACHET_DIRECTION_EMPLOI_PRESENT": null
}

- PTR_NUMERO_SERIE :
  numéro de série imprimé en bas de la couverture,
  après « N° de Série » ou isolé en bas à gauche.
  null si illisible.

- PTR_WILAYA :
  valeur après « Direction de l'Emploi de la Wilaya de : ».
  null si la ligne est vide.

- PTR_CACHET_DIRECTION_EMPLOI_PRESENT :
  true si un cachet officiel est visible sur cette zone.

Ne pas extraire le contenu des extraits de loi imprimés à droite.
"""

PROMPTS_EXTRACTION = {
    "ENGAGEMENT_DOMICILIATION": PROMPT_ENGAGEMENT,
    "CONTRAT_TRAVAIL": PROMPT_CONTRAT,
    "CONTRAT_SPECIFIQUE": PROMPT_CONTRAT_SPECIFIQUE,
    "TITRE_TRAVAIL": PROMPT_TITRE_TRAVAIL,
    "PERMIS_TRAVAIL_COUVERTURE": PROMPT_PERMIS_COUVERTURE,
}

TYPES_VALIDES = set(PROMPTS_EXTRACTION) | {"AUTRE"}


def champs_attendus_depuis_prompt(prompt):
    """
    Récupère la liste des clés depuis le squelette JSON du prompt.
    Évite de maintenir une seconde liste qui divergerait des prompts.
    """
    match = re.search(r"\{[^{}]*\}", prompt, flags=re.S)
    if not match:
        return []
    bloc = match.group(0)
    try:
        return list(json.loads(bloc).keys())
    except Exception:
        return re.findall(r'"([A-Z0-9_]+)"\s*:', bloc)


CHAMPS_ATTENDUS = {
    doc_type: champs_attendus_depuis_prompt(prompt)
    for doc_type, prompt in PROMPTS_EXTRACTION.items()
}

# ---------------------------------------------------------------------
# Stratégies d'extraction par type de page
# ---------------------------------------------------------------------
# Chaque stratégie est essayée dans l'ordre. On s'arrête dès que le taux
# de remplissage atteint SEUIL_REMPLISSAGE_OK. Les résultats successifs
# sont fusionnés : une passe ne peut que compléter des champs vides,
# jamais écraser une valeur déjà trouvée.
#
# crop = (haut, bas, gauche, droite) en fractions de la page.

STRATEGIES_EXTRACTION = {
    # Planche « permis de travail » : le titre de travail occupe la moitié
    # haute. On le recadre d'emblée sur la frontière détectée, puis on
    # isole chaque colonne si le remplissage reste insuffisant.
    "TITRE_TRAVAIL": [
        {"nom": "HD_BLOC_TITRE", "crop_dynamique": "titre"},
        {"nom": "HD_COLONNE_IDENTITE", "crop_dynamique": "colonne_identite"},
        {"nom": "HD_COLONNE_POSTE", "crop_dynamique": "colonne_poste"},
        {"nom": "PAGE_ENTIERE_HD", "crop": (0.00, 1.00, 0.00, 1.00)},
    ],
    # Couverture : bloc bas de la même planche.
    "PERMIS_TRAVAIL_COUVERTURE": [
        {"nom": "HD_BLOC_COUVERTURE", "crop_dynamique": "couverture"},
        {"nom": "PAGE_ENTIERE_HD", "crop": (0.00, 1.00, 0.00, 1.00)},
    ],
    # Documents dactylographiés : l'image standard suffit.
    "ENGAGEMENT_DOMICILIATION": [{"nom": "STANDARD"}],
    "CONTRAT_TRAVAIL": [{"nom": "STANDARD"}],
    "CONTRAT_SPECIFIQUE": [
        {"nom": "STANDARD"},
        {"nom": "HD_BLOC_CENTRAL", "crop": (0.05, 0.70, 0.00, 1.00)},
    ],
}

print("✅ Prompts V7.1 chargés")
print("   Types gérés          :", ", ".join(sorted(PROMPTS_EXTRACTION)))
print("   Champs TITRE_TRAVAIL :", len(CHAMPS_ATTENDUS["TITRE_TRAVAIL"]))
print("   Champs CONTRAT_SPEC. :", len(CHAMPS_ATTENDUS["CONTRAT_SPECIFIQUE"]))


## 8. Normalisation technique

In [ ]:

NULL_VALUES = {"", "NULL", "NONE", "N/A", "NA", "NEANT", "NÉANT", "ILLISIBLE"}

def clean_raw_value(value):
    if value is None or isinstance(value, bool):
        return value
    text = str(value).strip()
    return None if text.upper() in NULL_VALUES else text

def clean_raw_dict(data):
    return {k: clean_raw_value(v) for k, v in data.items()} if isinstance(data, dict) else {}

def normalize_amount(value):
    if value is None:
        return None
    text = re.sub(r"[^0-9,.\-]", "", str(value).replace("\xa0", " ").strip())
    if not text:
        return None
    if "," in text and "." in text:
        text = text.replace(".", "").replace(",", ".") if text.rfind(",") > text.rfind(".") else text.replace(",", "")
    elif "," in text:
        text = text.replace(",", ".")
    try:
        return round(float(text), 2)
    except Exception:
        return None

def parse_date(value):
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        return value.date()
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    parsed = pd.to_datetime(str(value).strip(), dayfirst=True, errors="coerce")
    return None if pd.isna(parsed) else parsed.date()

def normalize_text(value):
    if value is None:
        return ""
    text = re.sub(r"[^\w\s]", " ", str(value).upper())
    return re.sub(r"\s+", " ", text).strip()

def name_similarity(a, b):
    from difflib import SequenceMatcher
    a, b = normalize_text(a), normalize_text(b)
    return SequenceMatcher(None, a, b).ratio() if a and b else 0.0


def normalize_dom_reference(value, date_domiciliation=None):
    """
    Format exact : 271901AAAAT40NNNNNDZD
    """
    if value is None:
        return None

    raw = str(value).strip().upper()
    compact = re.sub(r"[^A-Z0-9]", "", raw)

    if re.fullmatch(r"271901\d{4}[1-4]40\d{5}[A-Z]{3}", compact):
        return compact

    parts = [p.strip().upper() for p in re.split(r"[|;/\\]+", raw) if p.strip()]
    if len(parts) >= 5:
        prefix = re.sub(r"\D", "", parts[0])
        fixed_code = re.sub(r"\D", "", parts[2])
        sequence = re.sub(r"\D", "", parts[3]).zfill(5)
        currency = re.sub(r"[^A-Z]", "", parts[4]) or "DZD"
        match_yq = re.search(r"(\d{4})\D*([1-4])", parts[1])
        if prefix == "271901" and fixed_code == "40" and match_yq:
            return f"271901{match_yq.group(1)}{match_yq.group(2)}40{sequence[:5]}{currency[:3]}"

    match_short = re.search(
        r"(\d{4})\D*([1-4])\D*40\D*(\d{1,5})(?:\D*([A-Z]{3}))?",
        raw
    )
    if match_short:
        year = match_short.group(1)
        quarter = match_short.group(2)
        sequence = match_short.group(3).zfill(5)
        currency = match_short.group(4) or "DZD"
        return f"271901{year}{quarter}40{sequence}{currency}"

    return compact or None


def dom_reference_is_valid(value):
    return bool(value and re.fullmatch(r"271901\d{4}[1-4]40\d{5}[A-Z]{3}", str(value)))

def first_not_null(*values):
    return next((v for v in values if v not in (None, "", "NULL")), None)

def safe_read_excel(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        frames = []
        xls = pd.ExcelFile(path)
        for sheet in xls.sheet_names:
            df = pd.read_excel(path, sheet_name=sheet)
            if not df.empty:
                df["_SOURCE_SHEET"] = sheet
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else None
    except Exception as exc:
        print(f"⚠️ Lecture impossible {path.name}: {exc}")
        return None

def resolve_column(df, aliases):
    if df is None:
        return None
    normalized = {normalize_text(c): c for c in df.columns}
    for alias in aliases:
        if normalize_text(alias) in normalized:
            return normalized[normalize_text(alias)]
    for alias in aliases:
        target = normalize_text(alias)
        for key, original in normalized.items():
            if target in key or key in target:
                return original
    return None

ALIASES = {
    "numero_dom": ["Numéro de domiciliation", "Numero de domiciliation", "N° domiciliation"],
    "date_dom": ["Date domiciliation", "Date de domiciliation", "Date demande", "Request Decision Date"],
    "nom_client": ["Nom complet/Raison social", "Nom complet/Raison sociale", "Nom du Fournisseur/Client", "Name", "Nom"],
    "numero_client": ["Identifiant client", "Numero client", "N° client", "Code client"],
    "date_debut": ["Date début du contrat", "Date debut du contrat"],
    "date_fin": ["Date fin de contrat", "Date de fin du contrat"],
    "reference": ["Référence", "Reference"],
}

def prepare_reference(df, source):
    if df is None or df.empty:
        return None
    out = pd.DataFrame()
    out["SOURCE_MATCH"] = source
    out["SOURCE_SHEET"] = df.get("_SOURCE_SHEET")
    for key, aliases in ALIASES.items():
        col = resolve_column(df, aliases)
        out[key] = df[col] if col else None
    out["numero_dom_normalise"] = out.apply(lambda r: normalize_dom_reference(r.get("numero_dom"), r.get("date_dom")), axis=1)
    out["date_debut_parse"] = out["date_debut"].apply(parse_date)
    out["date_fin_parse"] = out["date_fin"].apply(parse_date)
    return out


def _first_non_empty(series):
    values = [
        v for v in series.tolist()
        if v is not None and not (isinstance(v, float) and pd.isna(v)) and str(v).strip() != ""
    ]
    return values[0] if values else None


def build_reference_table():
    """
    DOM est le référentiel principal.
    PREDOM est joint à gauche uniquement pour enrichir DOM avec :
    - date début du contrat ;
    - date fin du contrat ;
    - référence PREDOM.

    Une ligne DOM + une ligne PREDOM portant le même numéro DOM
    représentent un seul dossier, et non deux candidats.
    """
    dom = prepare_reference(safe_read_excel(DOM_REFERENCE_EXCEL), "DOM")
    predom = prepare_reference(safe_read_excel(PREDOM_REFERENCE_EXCEL), "PREDOM")

    if dom is None or dom.empty:
        return None

    # Une seule ligne principale par numéro DOM.
    # Les doublons DOM restent signalés pour éviter une attribution automatique risquée.
    dom = dom.copy()
    dom["DOM_MATCH_COUNT"] = dom.groupby("numero_dom_normalise")["numero_dom_normalise"].transform("size")

    if predom is None or predom.empty:
        dom["date_debut_predom"] = None
        dom["date_fin_predom"] = None
        dom["reference_predom"] = None
        dom["PREDOM_MATCH_COUNT"] = 0
        return dom

    predom = predom.copy()

    # Consolidation PREDOM par numéro DOM.
    # On ne choisit pas arbitrairement entre plusieurs valeurs différentes :
    # le nombre de lignes est conservé dans PREDOM_MATCH_COUNT.
    predom_grouped = (
        predom.groupby("numero_dom_normalise", dropna=False)
        .agg(
            date_debut_predom=("date_debut", _first_non_empty),
            date_fin_predom=("date_fin", _first_non_empty),
            date_debut_predom_parse=("date_debut_parse", _first_non_empty),
            date_fin_predom_parse=("date_fin_parse", _first_non_empty),
            reference_predom=("reference", _first_non_empty),
            PREDOM_MATCH_COUNT=("numero_dom_normalise", "size"),
        )
        .reset_index()
    )

    reference = dom.merge(
        predom_grouped,
        on="numero_dom_normalise",
        how="left",
        validate="many_to_one",
    )

    reference["PREDOM_MATCH_COUNT"] = (
        reference["PREDOM_MATCH_COUNT"].fillna(0).astype(int)
    )

    # Pour le matching par période, les dates PREDOM sont prioritaires.
    # Si elles sont absentes, on utilise les dates disponibles dans DOM.
    reference["date_debut_match"] = reference["date_debut_predom_parse"].where(
        reference["date_debut_predom_parse"].notna(),
        reference["date_debut_parse"],
    )
    reference["date_fin_match"] = reference["date_fin_predom_parse"].where(
        reference["date_fin_predom_parse"].notna(),
        reference["date_fin_parse"],
    )

    return reference

print("✅ Helpers V6.3 chargés : DOM principal + enrichissement PREDOM")


# ---------------------------------------------------------------------
# V7.1 : relecture de la ligne de salaire du contrat spécifique
# ---------------------------------------------------------------------

# Un montant algérien peut s'écrire 506,471.38 / 506 471,38 / 506471.38
MOTIF_MONTANT = r"\d[\d\s\u00a0.,]*\d|\d"

# Variantes rencontrées : « au lieu de », « au lieu du », « en lieu de »
MOTIF_AU_LIEU_DE = r"\ben\s+lieu\s+de\b|\bau\s+lieu\s+d[eu]\b"


def split_ligne_salaire(ligne):
    """
    Décompose « Salaire mensuel de base net : X au lieu de Y ».

    Retourne les montants normalisés :
      - nouveau : montant avant « au lieu de » (salaire applicable) ;
      - ancien  : montant après « au lieu de » (salaire précédent).

    Sert de filet de sécurité lorsque le modèle a recopié la ligne
    complète sans isoler correctement les deux montants.
    """
    vide = {"nouveau": None, "ancien": None, "mention_presente": False}

    if not ligne:
        return vide

    texte = str(ligne).replace("\xa0", " ").strip()
    separateur = re.search(MOTIF_AU_LIEU_DE, texte, flags=re.I)

    if not separateur:
        montants = re.findall(MOTIF_MONTANT, texte)
        return {
            "nouveau": normalize_amount(montants[0]) if montants else None,
            "ancien": None,
            "mention_presente": False,
        }

    avant = texte[: separateur.start()]
    apres = texte[separateur.end():]

    montants_avant = re.findall(MOTIF_MONTANT, avant)
    montants_apres = re.findall(MOTIF_MONTANT, apres)

    return {
        # Dernier montant avant la mention : le libellé peut contenir des
        # chiffres parasites en début de ligne.
        "nouveau": normalize_amount(montants_avant[-1]) if montants_avant else None,
        "ancien": normalize_amount(montants_apres[0]) if montants_apres else None,
        "mention_presente": True,
    }


print("✅ Relecture de la ligne de salaire (augmentation) OK")


## 9. Consolidation des documents et informations KYC

In [ ]:

def build_page_row(pdf_name, page_record):
    row = {
        "FICHIER": pdf_name,
        "PAGE": page_record.get("page_num"),
        "TYPE_DOCUMENT": page_record.get("doc_type"),
        "TITRE_DETECTE": page_record.get("titre_detecte"),
        "CONFIANCE_CLASSIFICATION": page_record.get("classification_confidence"),
        "CLASSIFICATION_REQUALIFIEE": page_record.get("classification_requalifiee"),
        "BLOC_IDENTITE_PRESENT": page_record.get("bloc_identite_present"),
        "STATUT_EXTRACTION": page_record.get("extraction_status"),
        "TAUX_REMPLISSAGE": page_record.get("extraction_taux_remplissage"),
        "STRATEGIES_UTILISEES": " > ".join(
            s.get("nom", "")
            for s in (page_record.get("extraction_strategies") or [])
        ) or None,
        "ERREUR_EXTRACTION": page_record.get("extraction_error"),
    }
    row.update(page_record.get("raw_data") or {})
    return row

def consolidate_dossier(pdf_name, records):
    row = {
        "FICHIER": pdf_name,
        "NB_PAGES": len(records),
        "TYPES_DOCUMENTS": " | ".join(str(r.get("doc_type")) for r in records),
    }
    pages_by_type = defaultdict(list)
    for record in records:
        pages_by_type[record.get("doc_type")].append(str(record.get("page_num")))
        for key, value in (record.get("raw_data") or {}).items():
            if row.get(key) in (None, ""):
                row[key] = value
    for doc_type, pages in pages_by_type.items():
        row[f"PAGES_{doc_type}"] = ",".join(pages)

    for field in AMOUNT_FIELDS:
        if field in row:
            row[field + "_RAW"] = row[field]
            row[field] = normalize_amount(row[field])

    raw_ref = first_not_null(row.get("CTR_REFERENCE_DOMICILIATION"))
    row["REFERENCE_DOM_EXTRAITE_RAW"] = raw_ref
    row["REFERENCE_DOM_EXTRAITE_NORMALISEE"] = normalize_dom_reference(raw_ref)
    row["REFERENCE_DOM_FORMAT_VALIDE"] = dom_reference_is_valid(row["REFERENCE_DOM_EXTRAITE_NORMALISEE"])
    row["NOM_CLIENT_REFERENCE"] = first_not_null(
        row.get("DOM_NOM_RAISON_SOCIAL_CLIENT"),
        row.get("CTR_NOM_PRENOM_TRAVAILLEUR"),
        row.get("CTS_NOM_PRENOM_TRAVAILLEUR"),
        " ".join(x for x in [str(row.get("TTR_NOM") or "").strip(), str(row.get("TTR_PRENOM") or "").strip()] if x) or None,
    )
    row["NUMERO_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_NUMERO_CONTRAT"), row.get("CTR_REFERENCE_DOCUMENT"), row.get("CTS_REFERENCE_DOCUMENT"))
    row["DATE_DEBUT_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_DATE_DEBUT_CONTRAT"), row.get("CTR_DATE_DEBUT_CONTRAT"), row.get("CTS_DATE_DEBUT_CONTRAT"))
    row["DATE_FIN_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_DATE_FIN_CONTRAT"), row.get("CTR_DATE_FIN_CONTRAT"), row.get("CTS_DATE_FIN_CONTRAT"))
    row["NUMERO_PERMIS_REFERENCE"] = first_not_null(
        row.get("TTR_NUMERO_PERMIS"),
        row.get("CTR_NUMERO_PERMIS_TRAVAIL"),
        row.get("CTS_NUMERO_PERMIS_TRAVAIL"),
        row.get("PTR_NUMERO_SERIE"),
    )

    # -----------------------------------------------------------------
    # V7.1 : augmentation de salaire
    # Ligne source du contrat spécifique :
    #   « Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29 »
    #   -> nouveau salaire = 506 471,38 ; ancien salaire = 479 274,29
    # -----------------------------------------------------------------
    salaire_nouveau = first_not_null(
        row.get("CTS_SALAIRE_NET"),
        row.get("CTR_SALAIRE_NET"),
        row.get("DOM_SALAIRE_NET_MENSUEL"),
    )
    salaire_ancien = row.get("CTS_SALAIRE_NET_ANCIEN")

    # Filet de sécurité : si le modèle n'a pas isolé les deux montants
    # mais a bien recopié la ligne complète, on relit la ligne brute.
    if salaire_ancien is None:
        secours = split_ligne_salaire(row.get("CTS_LIGNE_SALAIRE_BRUTE"))
        if secours["ancien"] is not None:
            salaire_ancien = secours["ancien"]
            row["CTS_SALAIRE_NET_ANCIEN"] = salaire_ancien
            row["SOURCE_AUGMENTATION"] = "RELECTURE_LIGNE_BRUTE"
            if salaire_nouveau is None and secours["nouveau"] is not None:
                salaire_nouveau = secours["nouveau"]
    elif salaire_ancien is not None:
        row["SOURCE_AUGMENTATION"] = "CHAMPS_MODELE"

    row["SALAIRE_NOUVEAU_AUGMENTE"] = salaire_nouveau
    row["SALAIRE_ANCIEN"] = salaire_ancien
    row["AUGMENTATION_DETECTEE"] = bool(
        salaire_ancien is not None
        and salaire_nouveau is not None
        and salaire_nouveau != salaire_ancien
    )

    if row["AUGMENTATION_DETECTEE"]:
        ecart = round(float(salaire_nouveau) - float(salaire_ancien), 2)
        row["MONTANT_AUGMENTATION"] = ecart
        row["SENS_VARIATION_SALAIRE"] = "AUGMENTATION" if ecart > 0 else "DIMINUTION"
        row["TAUX_AUGMENTATION_PCT"] = (
            round(ecart / float(salaire_ancien) * 100, 2)
            if float(salaire_ancien)
            else None
        )
    else:
        row["MONTANT_AUGMENTATION"] = None
        row["SENS_VARIATION_SALAIRE"] = None
        row["TAUX_AUGMENTATION_PCT"] = None
        row.setdefault("SOURCE_AUGMENTATION", None)

    # Cohérence entre l'engagement de domiciliation et le contrat
    # spécifique : le salaire domicilié doit être le NOUVEAU salaire.
    salaire_dom = row.get("DOM_SALAIRE_NET_MENSUEL")
    if salaire_dom is not None and salaire_nouveau is not None:
        row["ECART_SALAIRE_DOM_CONTRAT"] = round(
            float(salaire_dom) - float(salaire_nouveau), 2
        )
        row["COHERENCE_SALAIRE_DOM_CONTRAT"] = (
            abs(row["ECART_SALAIRE_DOM_CONTRAT"]) < 0.01
        )
    else:
        row["ECART_SALAIRE_DOM_CONTRAT"] = None
        row["COHERENCE_SALAIRE_DOM_CONTRAT"] = None

    # Alerte : le salaire domicilié correspond à l'ANCIEN salaire alors
    # qu'une augmentation est actée dans le contrat spécifique.
    row["ALERTE_DOM_SUR_ANCIEN_SALAIRE"] = bool(
        row["AUGMENTATION_DETECTEE"]
        and salaire_dom is not None
        and salaire_ancien is not None
        and abs(float(salaire_dom) - float(salaire_ancien)) < 0.01
    )

    # -----------------------------------------------------------------
    # V7.1 : identité consolidée, enrichie par le titre de travail
    # -----------------------------------------------------------------
    row["NOM_TRAVAILLEUR_REFERENCE"] = first_not_null(
        row.get("CTR_NOM_PRENOM_TRAVAILLEUR"),
        row.get("CTS_NOM_PRENOM_TRAVAILLEUR"),
        " ".join(
            x for x in [
                str(row.get("TTR_NOM") or "").strip(),
                str(row.get("TTR_PRENOM") or "").strip(),
            ] if x
        ) or None,
    )
    row["DATE_NAISSANCE_REFERENCE"] = first_not_null(
        row.get("CTR_DATE_NAISSANCE"),
        row.get("CTS_DATE_NAISSANCE"),
        row.get("TTR_DATE_NAISSANCE"),
    )
    row["NATIONALITE_REFERENCE"] = first_not_null(
        row.get("CTR_NATIONALITE"),
        row.get("CTS_NATIONALITE"),
        row.get("TTR_NATIONALITE"),
    )
    row["DATE_ENTREE_ALGERIE"] = row.get("TTR_DATE_ENTREE_ALGERIE")
    row["PERMIS_TRAVAIL_LU"] = bool(
        row.get("TTR_NOM") or row.get("TTR_NUMERO_PERMIS")
    )

    return row


def match_dossier(row, ref):
    """
    Matching sécurisé.

    1. Si le numéro DOM est extrait du PDF :
       correspondance exacte uniquement.

    2. Si aucun numéro DOM n'est extrait :
       recherche uniquement sur la période exacte du contrat
       (date début + date fin).

    3. Le numéro DOM n'est retenu que lorsqu'un seul candidat DOM
       est incontestable. Sinon, NUMERO_DOM_RETENU reste vide.
    """
    result = {
        "MATCH_SOURCE": None,
        "MATCH_METHOD": None,
        "MATCH_SCORE": 0.00,
        "MATCH_STATUS": "AUCUN_MATCH",
        "MATCH_CANDIDATES_COUNT": 0,
        "NUMERO_CLIENT_RETENU": None,
        "NUMERO_DOM_RETENU": None,
        "DATE_DOM_RETENUE": None,
        "REFERENCE_EXTERNE_RETENUE": None,
        "REFERENCE_PREDOM_RETENUE": None,
        "DATE_DEBUT_CONTRAT_PREDOM": None,
        "DATE_FIN_CONTRAT_PREDOM": None,
        "PREDOM_TROUVEE": False,
    }

    if ref is None or ref.empty:
        result["MATCH_STATUS"] = "REFERENTIEL_ABSENT"
        return result

    dom_ref = row.get("REFERENCE_DOM_EXTRAITE_NORMALISEE")

    # ---------------------------------------------------------
    # 1) Numéro DOM extrait : matching exact uniquement
    # ---------------------------------------------------------
    if dom_ref:
        exact = ref[ref["numero_dom_normalise"] == dom_ref].copy()
        result["MATCH_CANDIDATES_COUNT"] = int(len(exact))
        result["MATCH_METHOD"] = "NUMERO_DOM_EXACT"

        if len(exact) == 0:
            result["MATCH_STATUS"] = "NUMERO_DOM_NON_TROUVE"
            return result

        if len(exact) > 1 or int(exact.iloc[0].get("DOM_MATCH_COUNT") or 1) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_DOM"
            return result

        best = exact.iloc[0]

    # ---------------------------------------------------------
    # 2) Numéro DOM absent : période exacte et candidat unique
    # ---------------------------------------------------------
    else:
        start_date = parse_date(row.get("DATE_DEBUT_CONTRAT_REFERENCE"))
        end_date = parse_date(row.get("DATE_FIN_CONTRAT_REFERENCE"))

        if not start_date or not end_date:
            result["MATCH_STATUS"] = "DATES_CONTRAT_INSUFFISANTES"
            result["MATCH_METHOD"] = "PERIODE_CONTRAT_EXACTE"
            return result

        period_matches = ref[
            (ref["date_debut_match"] == start_date)
            & (ref["date_fin_match"] == end_date)
        ].copy()

        result["MATCH_CANDIDATES_COUNT"] = int(len(period_matches))
        result["MATCH_METHOD"] = "PERIODE_CONTRAT_EXACTE"

        if len(period_matches) == 0:
            result["MATCH_STATUS"] = "AUCUN_MATCH_PERIODE"
            return result

        if len(period_matches) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_PERIODE"
            return result

        best = period_matches.iloc[0]

        # Sécurité supplémentaire : pas d'attribution si le DOM principal
        # contient lui-même plusieurs lignes pour ce numéro.
        if int(best.get("DOM_MATCH_COUNT") or 1) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_DOM"
            return result

    # ---------------------------------------------------------
    # 3) Attribution seulement après match unique et sûr
    # ---------------------------------------------------------
    predom_count = int(best.get("PREDOM_MATCH_COUNT") or 0)

    result.update({
        "MATCH_SOURCE": "DOM",
        "MATCH_SCORE": 100.00,
        "MATCH_STATUS": (
            "DOMICILIATION_TROUVEE"
            if dom_ref
            else "MATCH_PERIODE_EXACTE"
        ),
        "NUMERO_CLIENT_RETENU": best.get("numero_client"),
        "NUMERO_DOM_RETENU": best.get("numero_dom_normalise") or best.get("numero_dom"),
        "DATE_DOM_RETENUE": best.get("date_dom"),
        "REFERENCE_EXTERNE_RETENUE": best.get("reference"),
        "REFERENCE_PREDOM_RETENUE": best.get("reference_predom"),
        "DATE_DEBUT_CONTRAT_PREDOM": best.get("date_debut_predom"),
        "DATE_FIN_CONTRAT_PREDOM": best.get("date_fin_predom"),
        "PREDOM_TROUVEE": predom_count >= 1,
    })

    # Plusieurs lignes PREDOM ne créent pas plusieurs candidats DOM,
    # mais sont signalées pour contrôle des données d'enrichissement.
    if predom_count > 1:
        result["MATCH_STATUS"] = "MATCH_DOM_TROUVE_PREDOM_MULTIPLE"

    return result

def month_segment_rows(row):
    start = parse_date(row.get("DATE_DEBUT_CONTRAT_REFERENCE"))
    end = parse_date(row.get("DATE_FIN_CONTRAT_REFERENCE"))
    if not start or not end or end < start:
        return []
    plafond = row.get("DOM_PART_TRANSFERABLE")
    rows, cursor = [], date(start.year, start.month, 1)
    while cursor <= end:
        days_month = calendar.monthrange(cursor.year, cursor.month)[1]
        month_end = date(cursor.year, cursor.month, days_month)
        seg_start, seg_end = max(start, cursor), min(end, month_end)
        if seg_start <= seg_end:
            # Règle métier :
            # - mois entièrement couvert : AAAA-MM, sans P1/P2 ;
            # - mois partiel commençant le 1er : P1 ;
            # - mois partiel commençant après le 1er : P2.
            mois_complet = (
                seg_start == cursor
                and seg_end == month_end
            )

            if mois_complet:
                part = None
                period = f"{cursor.year:04d}-{cursor.month:02d}"
            elif seg_start.day == 1:
                part = "P1"
                period = f"{cursor.year:04d}-{cursor.month:02d}P1"
            else:
                part = "P2"
                period = f"{cursor.year:04d}-{cursor.month:02d}P2"
            nb_days = (seg_end - seg_start).days + 1
            coef = round(nb_days / days_month, 8)
            rows.append({
                "FICHIER": row.get("FICHIER"),
                "NUMERO_DOMICILIATION": row.get("NUMERO_DOM_RETENU"),
                "DATE_DOMICILIATION": row.get("DATE_DOM_RETENUE"),
                "DATE_DOMICILIATION_SOURCE": "FICHIER_DOM" if row.get("DATE_DOM_RETENUE") else None,
                "NUMERO_CLIENT": row.get("NUMERO_CLIENT_RETENU"),
                "NOM_CLIENT": row.get("NOM_CLIENT_REFERENCE"),
                "NUMERO_CONTRAT": row.get("NUMERO_CONTRAT_REFERENCE"),
                "DATE_DEBUT_CONTRAT": start.isoformat(),
                "DATE_FIN_CONTRAT": end.isoformat(),
                "NUMERO_PERMIS_TRAVAIL": row.get("NUMERO_PERMIS_REFERENCE"),
                "PERIODE_TL": period, "MOIS_BASE": f"{cursor.year:04d}-{cursor.month:02d}",
                "PARTIE": part, "DATE_DEBUT_SEGMENT": seg_start.isoformat(),
                "DATE_FIN_SEGMENT": seg_end.isoformat(),
                "NB_JOURS_SEGMENT": round(float(nb_days), 2), "NB_JOURS_MOIS": round(float(days_month), 2),
                "COEFFICIENT_PRORATA": round(float(coef), 2),
                "SALAIRE_NET_REFERENCE": row.get("DOM_SALAIRE_NET_MENSUEL"),
                "TAUX_TRANSFERABLE_REFERENCE_RAW": row.get("DOM_TAUX_TRANSFERABLE"),
                "PLAFOND_MENSUEL_REFERENCE": plafond,
                "MONTANT_MAX_THEORIQUE": round(plafond * coef, 2) if plafond is not None else None,
                "MONTANT_AUTORISE_SAISI": None, "MONTANT_TRANSFERE": None,
                "SOLDE_RESTANT": None,
                "MATCH_STATUS": row.get("MATCH_STATUS"),
                "MATCH_METHOD": row.get("MATCH_METHOD"),
                "MATCH_SCORE": row.get("MATCH_SCORE"),
                "MATCH_CANDIDATES_COUNT": row.get("MATCH_CANDIDATES_COUNT"),
                "REFERENCE_PREDOM": row.get("REFERENCE_PREDOM_RETENUE"),
                # V7.1 : contexte augmentation, répété sur chaque période
                # pour que les contrôles mensuels Alteryx sachent sur quelle
                # base salariale le plafond a été calculé.
                "SALAIRE_ANCIEN": row.get("SALAIRE_ANCIEN"),
                "SALAIRE_NOUVEAU_AUGMENTE": row.get("SALAIRE_NOUVEAU_AUGMENTE"),
                "AUGMENTATION_DETECTEE": row.get("AUGMENTATION_DETECTEE"),
                "MONTANT_AUGMENTATION": row.get("MONTANT_AUGMENTATION"),
                "TAUX_AUGMENTATION_PCT": row.get("TAUX_AUGMENTATION_PCT"),
                "ALERTE_DOM_SUR_ANCIEN_SALAIRE": row.get("ALERTE_DOM_SUR_ANCIEN_SALAIRE"),
            })
        cursor = date(cursor.year + 1, 1, 1) if cursor.month == 12 else date(cursor.year, cursor.month + 1, 1)
    return rows

print("✅ Consolidation, matching sécurisé et planning V6.3 chargés")


In [ ]:

# Test technique P1/P2 de la V6
_demo = {
    "FICHIER": "demo.pdf",
    "DATE_DEBUT_CONTRAT_REFERENCE": "12/06/2025",
    "DATE_FIN_CONTRAT_REFERENCE": "11/06/2026",
    "DOM_PART_TRANSFERABLE": 442985.84,
    "DOM_SALAIRE_NET_MENSUEL": 466300.88,
    "DOM_TAUX_TRANSFERABLE": "95%",
    "NUMERO_CONTRAT_REFERENCE": "DEMO",
    "NOM_CLIENT_REFERENCE": "CLIENT DEMO",
    "NUMERO_PERMIS_REFERENCE": "PERMIS-DEMO",
    "NUMERO_DOM_RETENU": "DOM-DEMO",
    "DATE_DOM_RETENUE": "01/01/2025",
    "NUMERO_CLIENT_RETENU": "CLIENT-001",
    "MATCH_STATUS": "TEST",
}

_demo_rows = month_segment_rows(_demo)

assert len(_demo_rows) == 13
assert _demo_rows[0]["PERIODE_TL"] == "2025-06P2"
assert _demo_rows[-1]["PERIODE_TL"] == "2026-06P1"
assert _demo_rows[0]["NB_JOURS_MOIS"] == 30
assert _demo_rows[0]["NB_JOURS_SEGMENT"] == 19
assert _demo_rows[-1]["NB_JOURS_SEGMENT"] == 11

print("✅ Test planning P1/P2 V6 réussi")
print(
    _demo_rows[0]["PERIODE_TL"],
    _demo_rows[0]["MONTANT_MAX_THEORIQUE"],
)
print(
    _demo_rows[-1]["PERIODE_TL"],
    _demo_rows[-1]["MONTANT_MAX_THEORIQUE"],
)


## 10. Gestion des mois scindés P1 / P2 et du planning TL

In [ ]:

assert normalize_dom_reference("271901|2026.1|40|00119|DZD") == "271901202614000119DZD"
assert normalize_dom_reference("2026.1.40-00119") == "271901202614000119DZD"
assert dom_reference_is_valid("271901202614000119DZD")
print("✅ Test format DOM réussi")


### Test de la règle P1 / P2 demandée

In [ ]:

print("Le planning est généré uniquement depuis la page ENGAGEMENT_DOMICILIATION.")


## 11. Classification et extraction d’un dossier

In [ ]:

def classify_pages(pages):
    records = []

    for start_idx in range(0, len(pages), GPU_BATCH_SIZE_CLASSIFICATION):
        batch = pages[start_idx:start_idx + GPU_BATCH_SIZE_CLASSIFICATION]
        outputs = ask_batch(
            PROMPT_CLASSIFICATION,
            [item["image"] for item in batch],
            MAX_NEW_TOKENS_CLASSIFICATION,
        )

        for page, output in zip(batch, outputs):
            parsed = parse_json_response(output["text"])
            doc_type = (
                parsed.get("type_document")
                or parsed.get("type")
                or "AUTRE"
            )
            confidence = parsed.get("confidence", 0)
            bloc_identite = bool(parsed.get("bloc_identite_present"))

            try:
                confidence = float(confidence or 0)
            except Exception:
                confidence = 0.0

            if doc_type not in TYPES_VALIDES:
                doc_type = "AUTRE"

            # -----------------------------------------------------------
            # Correctif V7.1 : la planche « permis de travail » porte deux
            # documents. Si le modèle signale un bloc d'identité tout en
            # classant la page en couverture, on force TITRE_TRAVAIL :
            # c'est le bloc identité qui porte les données exploitables.
            # -----------------------------------------------------------
            requalifie = False
            if bloc_identite and doc_type in ("PERMIS_TRAVAIL_COUVERTURE", "AUTRE"):
                doc_type = "TITRE_TRAVAIL"
                confidence = max(confidence, CLASSIFICATION_THRESHOLD)
                requalifie = True

            if confidence < CLASSIFICATION_THRESHOLD and not requalifie:
                doc_type = "AUTRE"

            records.append({
                "page_num": page["page_num"],
                "image": page["image"],
                "width": page["width"],
                "height": page["height"],
                "white_ratio": page["white_ratio"],
                "doc_type": doc_type,
                "titre_detecte": parsed.get("titre_detecte"),
                "bloc_identite_present": bloc_identite,
                "classification_requalifiee": requalifie,
                "classification_confidence": confidence,
                "classification_raw_text": output["text"],
                "classification_tokens_in": output["tokens_in"],
                "classification_tokens_out": output["tokens_out"],
                "classification_elapsed_s": output["elapsed_s"],
                "raw_data": {},
                "extraction_status": "NON_LANCEE",
                "extraction_error": None,
                "extraction_raw_text": None,
                "extraction_strategies": None,
                "extraction_taux_remplissage": 0.0,
                "extraction_tokens_in": 0,
                "extraction_tokens_out": 0,
                "extraction_elapsed_s": 0.0,
            })

    return records


def extract_one_record(record, pdf_path):
    """
    Extraction d'une page avec escalade progressive.

    Les stratégies sont appliquées dans l'ordre défini par
    STRATEGIES_EXTRACTION. On s'arrête dès que le taux de remplissage
    atteint SEUIL_REMPLISSAGE_OK. Chaque passe ne peut que compléter des
    champs restés vides : une valeur déjà extraite n'est jamais écrasée.
    """
    doc_type = record["doc_type"]
    prompt = PROMPTS_EXTRACTION[doc_type]
    champs = CHAMPS_ATTENDUS.get(doc_type) or []
    strategies = STRATEGIES_EXTRACTION.get(doc_type) or [{"nom": "STANDARD"}]

    fusion = {}
    strategies_utilisees = []
    textes_bruts = []
    tokens_in = 0
    tokens_out = 0
    elapsed = 0.0

    # Sur la planche « permis de travail », la coupure entre les deux
    # documents est mesurée sur l'image plutôt que codée en dur : le
    # cadrage du photocopieur varie d'un dossier à l'autre.
    crops_planche = None
    if doc_type in TYPES_HAUTE_DEFINITION:
        crops_planche = crops_planche_permis(record["image"])
        record["frontiere_planche"] = crops_planche["frontiere"]

    for rang, strategie in enumerate(strategies):
        if rang > 0 and taux_remplissage(fusion, champs) >= SEUIL_REMPLISSAGE_OK:
            break

        crop = strategie.get("crop")

        # Un recadrage nommé est résolu dynamiquement sur la planche.
        cle_dynamique = strategie.get("crop_dynamique")
        if cle_dynamique and crops_planche:
            crop = crops_planche[cle_dynamique]

        if crop is None:
            image = record["image"]
        else:
            image = render_page_region(
                pdf_path,
                record["page_num"] - 1,
                zoom=strategie.get("zoom", PDF_ZOOM_HAUTE_DEF),
                max_side=strategie.get("max_side", IMAGE_MAX_SIZE_HAUTE_DEF),
                crop=crop,
            )

        output = ask_single(prompt, image, MAX_NEW_TOKENS_EXTRACTION)
        parsed = clean_raw_dict(parse_json_response(output["text"]))

        nouveaux = 0
        for cle, valeur in parsed.items():
            if fusion.get(cle) in (None, "") and valeur not in (None, ""):
                fusion[cle] = valeur
                nouveaux += 1

        tokens_in += output["tokens_in"]
        tokens_out += output["tokens_out"]
        elapsed += output["elapsed_s"]
        textes_bruts.append(f"[{strategie['nom']}] {output['text']}")
        strategies_utilisees.append({
            "nom": strategie["nom"],
            "crop": crop,
            "champs_ajoutes": nouveaux,
            "taux_apres": taux_remplissage(fusion, champs),
        })

    # Les clés jamais renseignées restent présentes et nulles, pour que le
    # schéma de sortie soit stable d'un dossier à l'autre.
    for cle in champs:
        fusion.setdefault(cle, None)

    record["raw_data"] = fusion
    record["extraction_taux_remplissage"] = taux_remplissage(fusion, champs)
    record["extraction_strategies"] = strategies_utilisees
    record["extraction_raw_text"] = "\n\n".join(textes_bruts)
    record["extraction_tokens_in"] = tokens_in
    record["extraction_tokens_out"] = tokens_out
    record["extraction_elapsed_s"] = round(elapsed, 3)
    record["extraction_status"] = (
        "OK"
        if record["extraction_taux_remplissage"] >= SEUIL_REMPLISSAGE_OK
        else ("PARTIELLE" if fusion else "JSON_VIDE")
    )

    return record


def extract_classified_pages(records, pdf_path):
    for record in records:
        if record["doc_type"] not in PROMPTS_EXTRACTION:
            record["extraction_status"] = "NON_APPLICABLE"
            continue

        try:
            extract_one_record(record, pdf_path)
        except Exception as exc:
            record["extraction_status"] = "ERREUR"
            record["extraction_error"] = repr(exc)

    return records


def process_pdf(pdf_path, verbose=True):
    t0 = time.time()

    if verbose:
        log(f"📁 {pdf_path.name}")

    pages = pdf_to_pages(pdf_path)
    if not pages:
        raise ValueError(f"Aucune page détectée dans {pdf_path.name}")

    records = classify_pages(pages)
    records = extract_classified_pages(records, pdf_path)

    page_rows = [
        build_page_row(pdf_path.name, record)
        for record in records
    ]

    dossier_row = consolidate_dossier(pdf_path.name, records)
    planning = []

    tokens_in = sum(
        record.get("classification_tokens_in", 0)
        + record.get("extraction_tokens_in", 0)
        for record in records
    )
    tokens_out = sum(
        record.get("classification_tokens_out", 0)
        + record.get("extraction_tokens_out", 0)
        for record in records
    )

    elapsed_total = round(time.time() - t0, 3)

    dossier_row.update({
        "STATUT_TRAITEMENT_PIPELINE": "TRAITE_NOUVEAU",
        "TEMPS_ECOULE_DOSSIER_S": round(elapsed_total, 2),
        "TOKENS_IN_DOSSIER": int(tokens_in),
        "TOKENS_OUT_DOSSIER": int(tokens_out),
        "TOKENS_TOTAL_DOSSIER": int(tokens_in + tokens_out),
    })

    dossier = {
        "source_file": pdf_path.name,
        "source_sha256": sha256_file(pdf_path),
        "pipeline_version": PIPELINE_VERSION,
        "stats": {
            "pages": len(pages),
            "tokens_in": tokens_in,
            "tokens_out": tokens_out,
            "tokens_total": tokens_in + tokens_out,
            "elapsed_s": elapsed_total,
        },
        "page_records": [
            {
                key: value
                for key, value in record.items()
                if key != "image"
            }
            for record in records
        ],
        "page_rows": page_rows,
        "dossier_row": dossier_row,
        "planning_tl": planning,
    }

    checkpoint = canonical_checkpoint_path(pdf_path)
    with open(checkpoint, "w", encoding="utf-8") as f:
        json.dump(dossier, f, ensure_ascii=False, indent=2, default=str)

    return dossier


print("✅ Classification V7.1 (requalification planche permis) OK")
print("✅ Extraction V7.1 avec escalade et fusion multi-recadrages OK")


## 12. Export Excel compatible Alteryx

In [ ]:

def ordered_columns(rows):
    preferred = [
        "FICHIER", "NB_PAGES", "TYPES_DOCUMENTS",
        "STATUT_TRAITEMENT_PIPELINE", "TEMPS_ECOULE_DOSSIER_S",
        "TOKENS_IN_DOSSIER", "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
        "REFERENCE_DOM_EXTRAITE_RAW", "REFERENCE_DOM_EXTRAITE_NORMALISEE",
        "NUMERO_DOM_RETENU", "DATE_DOM_RETENUE",
            "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
            "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE", "NUMERO_CLIENT_RETENU",
        "NOM_CLIENT_REFERENCE", "NUMERO_CONTRAT_REFERENCE",
        "DATE_DEBUT_CONTRAT_REFERENCE", "DATE_FIN_CONTRAT_REFERENCE",
        "NUMERO_PERMIS_REFERENCE", "MATCH_SOURCE", "MATCH_METHOD",
        "MATCH_SCORE", "MATCH_STATUS", "MATCH_CANDIDATES_COUNT",
        "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
        "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE",
        # --- V7.1 : augmentation de salaire ---
        "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE",
        "AUGMENTATION_DETECTEE", "SENS_VARIATION_SALAIRE",
        "MONTANT_AUGMENTATION", "TAUX_AUGMENTATION_PCT",
        "SOURCE_AUGMENTATION", "CTS_LIGNE_SALAIRE_BRUTE",
        "ECART_SALAIRE_DOM_CONTRAT", "COHERENCE_SALAIRE_DOM_CONTRAT",
        "ALERTE_DOM_SUR_ANCIEN_SALAIRE",
        # --- V7.1 : identité et permis de travail ---
        "NOM_TRAVAILLEUR_REFERENCE", "DATE_NAISSANCE_REFERENCE",
        "NATIONALITE_REFERENCE", "DATE_ENTREE_ALGERIE",
        "PERMIS_TRAVAIL_LU",
    ]
    cols = set().union(*(r.keys() for r in rows)) if rows else set()
    return [c for c in preferred if c in cols] + sorted(cols - set(preferred))

def sheet_from_rows(wb, title, rows, columns=None, amount_columns=None):
    ws = wb.create_sheet(title)
    columns = columns or ordered_columns(rows)
    if not columns:
        ws["A1"] = "Aucune donnée"
        return
    amount_columns = set(amount_columns or [])
    fill = PatternFill("solid", fgColor="1F4E78")
    font = Font(color="FFFFFF", bold=True, name="Arial", size=9)
    for j, name in enumerate(columns, 1):
        c = ws.cell(1, j, name); c.fill = fill; c.font = font
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for i, row in enumerate(rows, 2):
        for j, name in enumerate(columns, 1):
            value = row.get(name)
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            c = ws.cell(i, j, value)
            if name in amount_columns and isinstance(value, (int, float)):
                c.number_format = EXCEL_AMOUNT_FORMAT
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    for j, name in enumerate(columns, 1):
        ws.column_dimensions[get_column_letter(j)].width = 38 if "ADRESSE" in name else min(34, max(14, len(name) + 2))

def build_long_raw_rows(dossiers):
    rows = []
    for d in dossiers:
        for page in d.get("page_records", []):
            for field, value in (page.get("raw_data") or {}).items():
                rows.append({
                    "FICHIER": d.get("source_file"), "PAGE": page.get("page_num"),
                    "TYPE_DOCUMENT": page.get("doc_type"), "CHAMP": field,
                    "VALEUR_BRUTE": value,
                    "CONFIANCE_CLASSIFICATION": page.get("classification_confidence"),
                    "STATUT_EXTRACTION": page.get("extraction_status"),
                })
    return rows

COLONNES_MONTANT_V71 = {
    "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE",
    "MONTANT_AUGMENTATION", "ECART_SALAIRE_DOM_CONTRAT",
}


def create_excel(excel_path, dossiers, errors, reference_df=None):
    pages, dossier_rows, planning, matching = [], [], [], []
    for d in dossiers:
        pages.extend(d.get("page_rows", []))
        row = d.get("dossier_row") or {}
        dossier_rows.append(row)
        planning.extend(month_segment_rows(row))
        matching.append({k: row.get(k) for k in [
            "FICHIER", "STATUT_TRAITEMENT_PIPELINE",
            "TEMPS_ECOULE_DOSSIER_S", "TOKENS_IN_DOSSIER",
            "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
            "NOM_CLIENT_REFERENCE", "NUMERO_CONTRAT_REFERENCE",
            "DATE_DEBUT_CONTRAT_REFERENCE", "DATE_FIN_CONTRAT_REFERENCE",
            "REFERENCE_DOM_EXTRAITE_RAW", "REFERENCE_DOM_EXTRAITE_NORMALISEE",
            "MATCH_SOURCE", "MATCH_METHOD", "MATCH_SCORE", "MATCH_STATUS",
            "MATCH_CANDIDATES_COUNT", "NUMERO_CLIENT_RETENU",
            "NUMERO_DOM_RETENU", "DATE_DOM_RETENUE",
            "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
            "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE",
            "AUGMENTATION_DETECTEE", "ALERTE_DOM_SUR_ANCIEN_SALAIRE",
        ]})
    raw = build_long_raw_rows(dossiers)
    suivi = []
    for d in dossiers:
        stats = d.get("stats") or {}
        row = d.get("dossier_row") or {}
        suivi.append({
            "FICHIER": d.get("source_file"),
            "STATUT_TRAITEMENT_PIPELINE": row.get("STATUT_TRAITEMENT_PIPELINE"),
            "NB_PAGES": stats.get("pages"),
            "TEMPS_ECOULE_DOSSIER_S": row.get("TEMPS_ECOULE_DOSSIER_S"),
            "TOKENS_IN_DOSSIER": row.get("TOKENS_IN_DOSSIER"),
            "TOKENS_OUT_DOSSIER": row.get("TOKENS_OUT_DOSSIER"),
            "TOKENS_TOTAL_DOSSIER": row.get("TOKENS_TOTAL_DOSSIER"),
            "PIPELINE_VERSION_JSON": d.get("pipeline_version"),
            "SHA256": d.get("source_sha256"),
        })

    wb = Workbook(); wb.remove(wb.active)
    sheet_from_rows(wb, "DOSSIERS_DOMICILIATION", dossier_rows,
                    amount_columns=AMOUNT_FIELDS | COLONNES_MONTANT_V71)
    sheet_from_rows(wb, "SUIVI_TRAITEMENT", suivi, [
        "FICHIER", "STATUT_TRAITEMENT_PIPELINE", "NB_PAGES",
        "TEMPS_ECOULE_DOSSIER_S", "TOKENS_IN_DOSSIER",
        "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
        "PIPELINE_VERSION_JSON", "SHA256",
    ], amount_columns={"TEMPS_ECOULE_DOSSIER_S"})
    sheet_from_rows(wb, "PLANNING_TL", planning, amount_columns={
        "SALAIRE_NET_REFERENCE", "PLAFOND_MENSUEL_REFERENCE",
        "MONTANT_MAX_THEORIQUE", "MONTANT_AUTORISE_SAISI",
        "MONTANT_TRANSFERE", "SOLDE_RESTANT", "NB_JOURS_SEGMENT", "NB_JOURS_MOIS", "COEFFICIENT_PRORATA",
        "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE", "MONTANT_AUGMENTATION",
    })
    sheet_from_rows(wb, "PAGES_DOCUMENTS", pages)
    sheet_from_rows(wb, "EXTRACTION_BRUTE", raw, [
        "FICHIER", "PAGE", "TYPE_DOCUMENT", "CHAMP", "VALEUR_BRUTE",
        "CONFIANCE_CLASSIFICATION", "STATUT_EXTRACTION",
    ])
    sheet_from_rows(wb, "MATCHING_DOM", matching)
    sheet_from_rows(wb, "ERREURS", errors)
    if reference_df is not None:
        sheet_from_rows(wb, "REFERENTIEL_DOM_PREDOM", reference_df.where(pd.notna(reference_df), None).to_dict("records"))
    wb.save(excel_path)
    print(f"✅ Excel créé : {excel_path} | dossiers={len(dossier_rows)} | planning={len(planning)}")

print("✅ Export V6.5 chargé avec onglet SUIVI_TRAITEMENT")


In [ ]:

print(f"Nombre de PDF détectés : {len(pdfs)}")
if not pdfs:
    raise RuntimeError(
        f"Aucun PDF trouvé dans {INPUT_DIR}. "
        "Déposer les dossiers de domiciliation dans ce répertoire."
    )

diagnostic_errors = []
total_pages = 0

for p in pdfs:
    try:
        with fitz.open(str(p)) as doc:
            page_count = int(doc.page_count)
        total_pages += page_count
        print(
            f"{p.name} -> pages={page_count}, "
            f"taille={p.stat().st_size:,} octets"
        )
        if page_count <= 0:
            diagnostic_errors.append(f"{p.name}: 0 page")
    except Exception as exc:
        diagnostic_errors.append(f"{p.name}: {exc!r}")

if diagnostic_errors:
    raise RuntimeError(
        "Diagnostic PDF en erreur :\n- " + "\n- ".join(diagnostic_errors)
    )

print(f"✅ Diagnostic PDF : {len(pdfs)} fichier(s), {total_pages} page(s)")

# Nettoyage des checkpoints invalides.
# Les anciens checkpoints valides suffixés par hash seront migrés
# au moment de leur chargement vers un seul JSON canonique par PDF.
removed = 0
for json_file in JSON_DIR.glob("*.json"):
    try:
        dossier = json.loads(json_file.read_text(encoding="utf-8"))
        pages_checkpoint = int(dossier.get("stats", {}).get("pages", 0) or 0)
        records_checkpoint = dossier.get("page_records") or []
        if pages_checkpoint <= 0 or not records_checkpoint:
            json_file.unlink()
            removed += 1
            print(f"Checkpoint vide supprimé : {json_file.name}")
    except Exception:
        json_file.unlink()
        removed += 1
        print(f"Checkpoint illisible supprimé : {json_file.name}")

print(f"Checkpoints invalides supprimés : {removed}")


In [ ]:

# Test technique sur le premier PDF avant le traitement complet.
_test_pdf = pdfs[0]
_test_pages = pdf_to_pages(_test_pdf)

print(
    f"Test conversion : {_test_pdf.name} -> "
    f"{len(_test_pages)} page(s)"
)
print(
    "Dimensions première page :",
    _test_pages[0]["width"],
    "x",
    _test_pages[0]["height"],
)
print(
    "Ratio blanc première page :",
    _test_pages[0]["white_ratio"],
)

if len(_test_pages) == 0:
    raise RuntimeError("Le test de conversion PDF a retourné zéro page.")

# Libération immédiate des images du test.
del _test_pages
gc.collect()
print("✅ Test de conversion PDF réussi")


## 13. Exécution complète avec reprise automatique

In [ ]:

# Tests techniques du matching sécurisé V6.3
_test_ref = pd.DataFrame([
    {
        "numero_dom": "271901202614000119DZD",
        "numero_dom_normalise": "271901202614000119DZD",
        "date_dom": "15/01/2026",
        "numero_client": "CL001",
        "reference": "DOM-001",
        "reference_predom": "PREDOM-001",
        "date_debut_predom": "12/06/2025",
        "date_fin_predom": "11/06/2026",
        "date_debut_match": date(2025, 6, 12),
        "date_fin_match": date(2026, 6, 11),
        "DOM_MATCH_COUNT": 1,
        "PREDOM_MATCH_COUNT": 1,
    }
])

# Numéro DOM absent + période exacte unique : attribution autorisée
_test_row_unique = {
    "REFERENCE_DOM_EXTRAITE_NORMALISEE": None,
    "DATE_DEBUT_CONTRAT_REFERENCE": "12/06/2025",
    "DATE_FIN_CONTRAT_REFERENCE": "11/06/2026",
}
_test_match_unique = match_dossier(_test_row_unique, _test_ref)
assert _test_match_unique["MATCH_STATUS"] == "MATCH_PERIODE_EXACTE"
assert _test_match_unique["NUMERO_DOM_RETENU"] == "271901202614000119DZD"

# Aucun match : aucun numéro DOM ne doit être renseigné
_test_row_none = {
    "REFERENCE_DOM_EXTRAITE_NORMALISEE": None,
    "DATE_DEBUT_CONTRAT_REFERENCE": "01/01/2030",
    "DATE_FIN_CONTRAT_REFERENCE": "31/12/2030",
}
_test_match_none = match_dossier(_test_row_none, _test_ref)
assert _test_match_none["MATCH_STATUS"] == "AUCUN_MATCH_PERIODE"
assert _test_match_none["NUMERO_DOM_RETENU"] is None

# Plusieurs candidats : aucun numéro DOM ne doit être renseigné
_test_ref_multiple = pd.concat([_test_ref, _test_ref.assign(
    numero_dom="271901202624000120DZD",
    numero_dom_normalise="271901202624000120DZD",
)], ignore_index=True)
_test_match_multiple = match_dossier(_test_row_unique, _test_ref_multiple)
assert _test_match_multiple["MATCH_STATUS"] == "PLUSIEURS_CANDIDATS_PERIODE"
assert _test_match_multiple["NUMERO_DOM_RETENU"] is None

print("✅ Tests matching sécurisé V6.3 réussis")


In [ ]:

# Tests techniques V6.4 : mois complet sans P1/P2 et date DOM dans Planning_TL
_test_full_month = {
    "FICHIER": "demo.pdf",
    "NUMERO_DOM_RETENU": "271901202614000119DZD",
    "DATE_DOM_RETENUE": "15/01/2026",
    "NUMERO_CLIENT_RETENU": "CL001",
    "NOM_CLIENT_REFERENCE": "CLIENT TEST",
    "NUMERO_CONTRAT_REFERENCE": "CTR001",
    "DATE_DEBUT_CONTRAT_REFERENCE": "01/05/2026",
    "DATE_FIN_CONTRAT_REFERENCE": "31/05/2026",
    "NUMERO_PERMIS_REFERENCE": "PT001",
    "DOM_PART_TRANSFERABLE": 100000.00,
    "DOM_SALAIRE_NET_MENSUEL": 120000.00,
    "DOM_TAUX_TRANSFERABLE": "80%",
    "MATCH_STATUS": "DOMICILIATION_TROUVEE",
    "MATCH_METHOD": "NUMERO_DOM_EXACT",
    "MATCH_SCORE": 100.00,
    "MATCH_CANDIDATES_COUNT": 1,
    "REFERENCE_PREDOM_RETENUE": "PREDOM001",
}

_test_rows = month_segment_rows(_test_full_month)
assert len(_test_rows) == 1
assert _test_rows[0]["PERIODE_TL"] == "2026-05"
assert _test_rows[0]["PARTIE"] is None
assert _test_rows[0]["COEFFICIENT_PRORATA"] == 1.00
assert _test_rows[0]["DATE_DOMICILIATION"] == "15/01/2026"
assert _test_rows[0]["DATE_DOMICILIATION_SOURCE"] == "FICHIER_DOM"

print("✅ Tests V6.4 réussis : mois complet sans P1/P2 + date DOM dans Planning_TL")


In [ ]:

# Tests techniques V6.5 : un seul JSON par PDF et statistiques dossier
assert canonical_checkpoint_path(Path("07000-670909-202605-DOM.pdf")).name == \
       "07000-670909-202605-DOM.json"

_test_stats_dossier = {
    "dossier_row": {"FICHIER": "demo.pdf"},
    "stats": {
        "elapsed_s": 12.345,
        "tokens_in": 1000,
        "tokens_out": 250,
        "tokens_total": 1250,
    },
}
_test_stats_dossier = enrich_dossier_row_with_stats(
    _test_stats_dossier,
    "REPRIS_JSON_EXISTANT",
)
_test_row = _test_stats_dossier["dossier_row"]

assert _test_row["TEMPS_ECOULE_DOSSIER_S"] == 12.35
assert _test_row["TOKENS_IN_DOSSIER"] == 1000
assert _test_row["TOKENS_OUT_DOSSIER"] == 250
assert _test_row["TOKENS_TOTAL_DOSSIER"] == 1250
assert _test_row["STATUT_TRAITEMENT_PIPELINE"] == "REPRIS_JSON_EXISTANT"

print("✅ Tests V6.5 réussis : reprise JSON + temps + tokens")


In [ ]:

# Tests techniques V6.7 : suivi compact Domino
assert format_duration(65) == "01:05"
assert format_duration(3661) == "01:01:01"

print("✅ Tests V6.7 réussis : suivi compact Domino")


In [ ]:

# =====================================================================
# Tests techniques V7.1
# =====================================================================

# --- 1. Décomposition de la ligne de salaire ------------------------

_cas = split_ligne_salaire(
    "Salaire mensuel de base net :  506,471.38  au lieu de  479,274.29"
)
assert _cas["nouveau"] == 506471.38, _cas
assert _cas["ancien"] == 479274.29, _cas
assert _cas["mention_presente"] is True

# Variante avec espaces insécables et virgule décimale
_cas_fr = split_ligne_salaire(
    "Salaire mensuel de base net : 506 471,38 au lieu de 479 274,29"
)
assert _cas_fr["nouveau"] == 506471.38, _cas_fr
assert _cas_fr["ancien"] == 479274.29, _cas_fr

# Sans augmentation : un seul montant, aucun ancien salaire
_cas_simple = split_ligne_salaire("Salaire mensuel de base net : 466,300.88")
assert _cas_simple["nouveau"] == 466300.88, _cas_simple
assert _cas_simple["ancien"] is None
assert _cas_simple["mention_presente"] is False

# Ligne absente
assert split_ligne_salaire(None)["ancien"] is None

print("✅ Décomposition de la ligne de salaire OK")

# --- 2. Consolidation de l'augmentation -----------------------------

_rec_augmentation = [{
    "page_num": 1,
    "doc_type": "CONTRAT_SPECIFIQUE",
    "raw_data": {
        "CTS_SALAIRE_NET": "506,471.38",
        "CTS_SALAIRE_NET_ANCIEN": "479,274.29",
        "CTS_LIGNE_SALAIRE_BRUTE":
            "Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29",
        "CTS_NOM_PRENOM_TRAVAILLEUR": "YILDIRIM IBRAHIM",
    },
}, {
    "page_num": 2,
    "doc_type": "ENGAGEMENT_DOMICILIATION",
    "raw_data": {
        "DOM_SALAIRE_NET_MENSUEL": "506471,38",
        "DOM_PART_TRANSFERABLE": "481 147.81",
    },
}]

_row = consolidate_dossier("test_augmentation.pdf", _rec_augmentation)

assert _row["SALAIRE_NOUVEAU_AUGMENTE"] == 506471.38, _row["SALAIRE_NOUVEAU_AUGMENTE"]
assert _row["SALAIRE_ANCIEN"] == 479274.29, _row["SALAIRE_ANCIEN"]
assert _row["AUGMENTATION_DETECTEE"] is True
assert _row["SENS_VARIATION_SALAIRE"] == "AUGMENTATION"
assert _row["MONTANT_AUGMENTATION"] == 27197.09, _row["MONTANT_AUGMENTATION"]
assert _row["TAUX_AUGMENTATION_PCT"] == 5.67, _row["TAUX_AUGMENTATION_PCT"]
assert _row["COHERENCE_SALAIRE_DOM_CONTRAT"] is True
assert _row["ALERTE_DOM_SUR_ANCIEN_SALAIRE"] is False

print(
    "✅ Augmentation détectée : "
    f"{_row['SALAIRE_ANCIEN']} -> {_row['SALAIRE_NOUVEAU_AUGMENTE']} "
    f"(+{_row['MONTANT_AUGMENTATION']} / +{_row['TAUX_AUGMENTATION_PCT']}%)"
)

# --- 3. Alerte : domiciliation restée sur l'ancien salaire -----------

_rec_alerte = [{
    "page_num": 1,
    "doc_type": "CONTRAT_SPECIFIQUE",
    "raw_data": {
        "CTS_SALAIRE_NET": "506,471.38",
        "CTS_SALAIRE_NET_ANCIEN": "479,274.29",
    },
}, {
    "page_num": 2,
    "doc_type": "ENGAGEMENT_DOMICILIATION",
    "raw_data": {"DOM_SALAIRE_NET_MENSUEL": "479 274,29"},
}]

_row_alerte = consolidate_dossier("test_alerte.pdf", _rec_alerte)
assert _row_alerte["ALERTE_DOM_SUR_ANCIEN_SALAIRE"] is True
assert _row_alerte["COHERENCE_SALAIRE_DOM_CONTRAT"] is False

print("✅ Alerte domiciliation sur ancien salaire OK")

# --- 4. Relecture de secours quand le modèle n'isole pas les montants -

_rec_secours = [{
    "page_num": 1,
    "doc_type": "CONTRAT_SPECIFIQUE",
    "raw_data": {
        "CTS_SALAIRE_NET": "506,471.38",
        "CTS_SALAIRE_NET_ANCIEN": None,
        "CTS_LIGNE_SALAIRE_BRUTE":
            "Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29",
    },
}]

_row_secours = consolidate_dossier("test_secours.pdf", _rec_secours)
assert _row_secours["SALAIRE_ANCIEN"] == 479274.29, _row_secours["SALAIRE_ANCIEN"]
assert _row_secours["SOURCE_AUGMENTATION"] == "RELECTURE_LIGNE_BRUTE"

print("✅ Relecture de secours depuis la ligne brute OK")

# --- 5. Champs attendus et stratégies -------------------------------

assert "CTS_SALAIRE_NET_ANCIEN" in CHAMPS_ATTENDUS["CONTRAT_SPECIFIQUE"]
assert "TTR_NOM" in CHAMPS_ATTENDUS["TITRE_TRAVAIL"]
assert "TTR_DATE_ENTREE_ALGERIE" in CHAMPS_ATTENDUS["TITRE_TRAVAIL"]
assert "TTR_NUMERO_PERMIS" in CHAMPS_ATTENDUS["TITRE_TRAVAIL"]
assert len(STRATEGIES_EXTRACTION["TITRE_TRAVAIL"]) == 4

# Taux de remplissage
assert taux_remplissage({"A": 1, "B": None}, ["A", "B"]) == 0.5
assert taux_remplissage({}, []) == 1.0

print("✅ Champs attendus et stratégies d'escalade OK")

# --- 6. Recadrage ----------------------------------------------------

_img_test = Image.new("RGB", (1000, 2000), "white")
_haut = crop_region(_img_test, 0.00, 0.58)
assert _haut.size == (1000, 1160), _haut.size

_droite = crop_region(_img_test, 0.00, 0.58, 0.44, 1.00)
assert _droite.size == (560, 1160), _droite.size

print("✅ Recadrage par fractions OK")

# --- 7. Détection automatique de la frontière de planche --------------

# Planche synthétique : bloc haut, bande blanche, bloc bas.
_planche = Image.new("L", (800, 2000), 255)
_px = _planche.load()
for y in list(range(60, 880)) + list(range(1120, 1940)):
    for x in range(40, 760, 3):
        _px[x, y] = 0

_frontiere = detecter_frontiere_documents(_planche.convert("RGB"))
assert 0.44 <= _frontiere <= 0.52, _frontiere

_crops = crops_planche_permis(_planche.convert("RGB"))
assert _crops["titre"][0] == 0.0
assert _crops["titre"][1] > _crops["couverture"][0], "les blocs doivent se chevaucher"
assert _crops["colonne_identite"][2] == 0.44
assert _crops["couverture"][1] == 1.0

print(f"✅ Frontière détectée sur planche synthétique : {_frontiere}")

# Repli sur une image sans bande blanche franche
assert detecter_frontiere_documents(Image.new("RGB", (800, 1200), "white")) == FRONTIERE_DEFAUT
print(f"✅ Repli sur image sans coupure nette : {FRONTIERE_DEFAUT}")

# --- 8. Stratégies dynamiques ----------------------------------------

_cles_dynamiques = {
    s.get("crop_dynamique")
    for s in STRATEGIES_EXTRACTION["TITRE_TRAVAIL"]
    if s.get("crop_dynamique")
}
assert _cles_dynamiques <= set(_crops), _cles_dynamiques
assert "colonne_identite" in _cles_dynamiques
assert STRATEGIES_EXTRACTION["PERMIS_TRAVAIL_COUVERTURE"][0]["crop_dynamique"] == "couverture"

print("✅ Stratégies dynamiques cohérentes avec les recadrages")
print("\n✅ Tous les tests V7.1 sont passés")



In [ ]:

ram_free = psutil.virtual_memory().available / 1_000_000_000
log(f"RAM libre : {ram_free:.1f} GB")
reference_df = build_reference_table()
log(f"Référentiel externe : {0 if reference_df is None else len(reference_df)} ligne(s)")

all_dossiers, errors = [], []
nb_repris = 0
nb_nouveaux = 0
pipeline_start = time.time()

print_pipeline_header(len(pdfs))

for position, pdf_path in enumerate(pdfs, 1):
    try:
        dossier = load_existing_checkpoint(pdf_path)
        skipped = dossier is not None

        if skipped:
            dossier = enrich_dossier_row_with_stats(
                dossier,
                "REPRIS_JSON_EXISTANT",
            )
            nb_repris += 1
        else:
            dossier = process_pdf(pdf_path, verbose=False)
            dossier = enrich_dossier_row_with_stats(
                dossier,
                "TRAITE_NOUVEAU",
            )
            nb_nouveaux += 1

        all_dossiers.append(dossier)

        stats = dossier.get("stats") or {}
        print_compact_progress(
            position=position,
            total=len(pdfs),
            pdf_name=pdf_path.name,
            pages=stats.get("pages", 0),
            skipped=skipped,
            tokens_in=stats.get("tokens_in", 0),
            tokens_out=stats.get("tokens_out", 0),
            elapsed_s=stats.get("elapsed_s", 0),
            pipeline_start=pipeline_start,
        )

    except Exception as exc:
        errors.append({
            "FICHIER": pdf_path.name,
            "ETAPE": "PROCESS_PDF",
            "ERREUR": repr(exc),
            "DATE": datetime.now().isoformat(timespec="seconds"),
        })

        print(
            f"[{position}/{len(pdfs)}] "
            f"{pdf_path.name} | ERREUR | {exc!r}",
            flush=True,
        )
        log(f"[{position}/{len(pdfs)}] ❌ {pdf_path.name}: {exc}")

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

for dossier in all_dossiers:
    row = dossier.get("dossier_row") or {}
    row.update(match_dossier(row, reference_df))
    dossier["dossier_row"] = row

    # Mise à jour du JSON canonique avec le matching actuel,
    # sans relancer la classification ni l'extraction VLM.
    source_pdf = INPUT_DIR / dossier.get("source_file", "")
    if source_pdf.exists():
        canonical_checkpoint_path(source_pdf).write_text(
            json.dumps(dossier, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8",
        )

create_excel(EXCEL_PATH, all_dossiers, errors, reference_df)

with open(MASTER_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "pipeline_version": PIPELINE_VERSION,
        "dossiers": all_dossiers, "errors": errors,
    }, f, ensure_ascii=False, indent=2, default=str)


elapsed_pipeline = time.time() - pipeline_start
total_tokens_in = sum(
    int((d.get("stats") or {}).get("tokens_in", 0) or 0)
    for d in all_dossiers
)
total_tokens_out = sum(
    int((d.get("stats") or {}).get("tokens_out", 0) or 0)
    for d in all_dossiers
)

print(
    f"\nTerminé | total={len(all_dossiers)} "
    f"| traités={nb_nouveaux} "
    f"| skip={nb_repris} "
    f"| erreurs={len(errors)} "
    f"| IN={total_tokens_in:,} "
    f"| OUT={total_tokens_out:,} "
    f"| durée={format_duration(elapsed_pipeline)}",
    flush=True,
)

log(f"✅ Pipeline terminé | dossiers={len(all_dossiers)} | nouveaux={nb_nouveaux} | repris={nb_repris} | erreurs={len(errors)}")



## 14. Lecture des résultats

- `domiciliations_master.json` : référentiel consolidé avec les liens de renouvellement et tout le planning.
- `DOMICILIATIONS` : une ligne par contrat/domiciliation.
- `PLANNING_TL` : une ligne par période de transfert autorisée (mois partiels en `P1` / `P2`).
- `DOCUMENTS` : classification et extraction page par page.
- `CHAMPS_SOURCE` : traçabilité de toutes les valeurs brutes et normalisées.
- `ERREURS` : erreurs techniques.
- `PARAMETRES` : version du modèle et règles de génération.

### Colonnes ajoutées en V7.1

**Augmentation de salaire** (issues du contrat spécifique) :

| Colonne | Contenu |
|---|---|
| `CTS_LIGNE_SALAIRE_BRUTE` | la ligne recopiée telle quelle, pour contrôle |
| `SALAIRE_ANCIEN` | montant après « au lieu de » |
| `SALAIRE_NOUVEAU_AUGMENTE` | montant avant « au lieu de » — salaire applicable |
| `AUGMENTATION_DETECTEE` | vrai si les deux montants diffèrent |
| `SENS_VARIATION_SALAIRE` | `AUGMENTATION` ou `DIMINUTION` |
| `MONTANT_AUGMENTATION` | écart en DZD |
| `TAUX_AUGMENTATION_PCT` | écart en pourcentage de l'ancien salaire |
| `SOURCE_AUGMENTATION` | `CHAMPS_MODELE` ou `RELECTURE_LIGNE_BRUTE` |
| `ECART_SALAIRE_DOM_CONTRAT` | écart entre le salaire domicilié et le nouveau salaire |
| `COHERENCE_SALAIRE_DOM_CONTRAT` | vrai si l'engagement porte bien le nouveau salaire |
| `ALERTE_DOM_SUR_ANCIEN_SALAIRE` | **vrai si la domiciliation est restée sur l'ancien salaire** |

`ALERTE_DOM_SUR_ANCIEN_SALAIRE` est la colonne à surveiller : elle signale
un engagement de domiciliation calculé sur un salaire périmé alors que le
contrat acte une augmentation.

**Permis de travail** (issues du titre de travail, désormais lisible) :

| Colonne | Contenu |
|---|---|
| `PERMIS_TRAVAIL_LU` | vrai si le bloc identité a été extrait |
| `NOM_TRAVAILLEUR_REFERENCE` | identité consolidée sur les trois documents |
| `DATE_NAISSANCE_REFERENCE` | date de naissance consolidée |
| `NATIONALITE_REFERENCE` | nationalité consolidée |
| `DATE_ENTREE_ALGERIE` | disponible uniquement sur le titre de travail |
| `NUMERO_PERMIS_REFERENCE` | numéro du permis, priorité au titre de travail |

**Diagnostic d'extraction**, dans l'onglet `DOCUMENTS` :

| Colonne | Contenu |
|---|---|
| `TAUX_REMPLISSAGE` | part des champs attendus effectivement extraits |
| `STRATEGIES_UTILISEES` | recadrages successifs, par exemple `HD_BLOC_TITRE > HD_COLONNE_IDENTITE` |
| `CLASSIFICATION_REQUALIFIEE` | vrai si la planche a été requalifiée en `TITRE_TRAVAIL` |
| `BLOC_IDENTITE_PRESENT` | signalement par le modèle d'un bloc identité |

Un `TAUX_REMPLISSAGE` bas sur `TITRE_TRAVAIL` avec toutes les stratégies
consommées indique un scan trop dégradé pour l'extraction automatique.


In [ ]:

if EXCEL_PATH.exists():
    for sheet in ["DOSSIERS_DOMICILIATION", "PLANNING_TL", "MATCHING_DOM"]:
        df = pd.read_excel(EXCEL_PATH, sheet_name=sheet)
        print(f"{sheet}: {len(df)} ligne(s)")
        display(df.head(10))
else:
    print("Le fichier Excel n'a pas encore été généré.")
